# Bayesian Optimisation of Sodium-Ion Battery Formulations under Low-Data Experimental Conditions

Data-processing, statistical-analysis and BO pipeline for the dissertation (Methods 3, Results 4), in the same order as the write-up. The DoE itself (3.1) was built and evaluated in JMP, not reproduced here -- it just defines the dataset used from 3.2 onward.

Group names here match the dissertation: `DOE` (mine), `EXTRA1` (Isabel's), `EXTRA2` (Nadia's). Raw cell-code prefixes (ltl_doe/IF/P009) are unchanged, only the group label is renamed.

ICE is the only modelling target from here on. Retention was tried early on but dropped, see Future Plans in the dissertation for why.


## Libraries

In [ ]:
## Standard library / data handling
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## File selection (manual raw-data import)
from tkinter import Tk
from tkinter.filedialog import askopenfilenames

## Neware binary export reader
import NewareNDA

## Preprocessing / classical ML
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler, OrdinalEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import shap

##Statistics
from scipy.stats import pearsonr, spearmanr
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

## Gaussian Processes / Bayesian Optimisation (BoTorch, Phase 1: DOE-only)
import torch
from botorch.models import SingleTaskGP, MixedSingleTaskGP
from botorch.fit import fit_gpytorch_mll
from botorch.acquisition import LogExpectedImprovement
from gpytorch.mlls import ExactMarginalLogLikelihood
from gpytorch.kernels import ScaleKernel, RBFKernel, MaternKernel
from gpytorch.means import ConstantMean
from gpytorch.priors import GammaPrior

##Bayesian Optimisation across data groups (BayBE, Phase 3: cross-group benchmark)
from baybe.parameters import CategoricalParameter, NumericalDiscreteParameter
from baybe.searchspace import SearchSpace
from baybe.searchspace.discrete import SubspaceDiscrete
from baybe.targets import NumericalTarget
from baybe.objectives import SingleTargetObjective
from baybe.campaign import Campaign
from baybe.simulation import simulate_scenarios

## 3.2 Data Import and Master Log

Raw cycling data comes from the Neware cycler in different file formats depending on the
test protocol: flat `.csv`/`.xlsx` exports for the rate-test protocol, and binary
`.nda`/`.ndax` files (read via `NewareNDA`) for the longterm protocol. Each format needs
its own parsing step before the files can be combined into one dataset.

The steps below follow the sequence described in Methods 3.2: select and inspect the raw
files, harmonise column names and formats, identify charge/discharge cycles directly from
the recorded data (rather than trusting the Neware `Cycle Index`, which does not reliably
separate formation cycles from the main cycling loop in half-cells), and consolidate
everything into a single `master_log.csv`.

In [ ]:
def select_files():
    ##opens a file dialog and returns the selected file paths as a list
    root = Tk()
    root.withdraw()
    file_paths = askopenfilenames(
        title="Select a file",
        filetypes=[("Files", "*.xlsx *.xls *.csv *.nda *.ndax")],
    )
    if not file_paths:
        print("No file selected.")
        return None
    return list(file_paths)

In [ ]:
def normalize_column_names(columns):
    ## lowercases, strips, and replaces spaces with underscores -- used for both the tracker file (load_cell_track) and the raw Neware exports (organize_data), so the same harmonization logic is not repeated in two places
    normalized = []
    for name in columns:
        clean_name = name.strip().lower().replace(" ", "_")
        normalized.append(clean_name)
    return normalized

In [ ]:
def load_cell_track(path, id_column="cell_code"):
    ##loads the tracker file with all cells, their codes, and their design parameters
    try:
        if path.endswith(".csv"):
            df_track = pd.read_csv(path)
        elif path.endswith((".xlsx", ".xls")):
            df_track = pd.read_excel(path)
        else:
            raise ValueError("Wrong file format. Use .csv or .xls/.xlsx files.")
    except PermissionError:
        print("Can't open the file, it may be open. Try again.")
        return None

    df_track.columns = normalize_column_names(df_track.columns.tolist())

    if id_column not in df_track.columns:
        raise ValueError(f"Column '{id_column}' not found in the file.")

    print(f"Track: {len(df_track)} cells, columns: {list(df_track.columns)}")
    return df_track

In [ ]:
def load_neware_raw(path):
    ## reads a single raw Neware export and returns its dataframe. File format is inferred from the extension: flat csv/xlsx for the rate-test protocol, binary nda/ndax for the longterm protocol
    try:
        if path.endswith(".csv"):
            df_raw = pd.read_csv(path, sep=None, engine="python")
        elif path.endswith((".xlsx", ".xls")):
            df_raw = pd.read_excel(path, sheet_name="record")
        elif path.endswith((".nda", ".ndax")):
            df_raw = NewareNDA.read(path)
        else:
            raise ValueError("Wrong file format. Use .csv, .xls/.xlsx, .nda, or .ndax files.")
    except PermissionError:
        print("Can't open the file, it may be open. Try again.")
        return None

    print(f"Neware raw data loaded with {df_raw.shape[0]} rows and {df_raw.shape[1]} columns")
    return df_raw

In [ ]:
def extract_cell_code(filename, known_codes):
    ##  matches a raw data filename to a cell code from the tracker. Uses the longest code that is a valid prefix of the filename stem (so e.g. "P009-CEL-107" is preferred over a shorter accidental partial match), and requires the character right after the code to be non-alphanumeric, so "P009-CEL-1" does not accidentally match a filename that is really for "P009-CEL-10"
    stem = Path(filename).stem

    best_match = None
    best_match_length = 0
    for code_value in known_codes:
        code_str = str(code_value)

        if stem == code_str:
            ##exact match -> always the best possible case, return immediately
            return code_str

        if stem.startswith(code_str):
            next_char = stem[len(code_str)]
            if not next_char.isalnum():
                if len(code_str) > best_match_length:
                    best_match = code_str
                    best_match_length = len(code_str)

    return best_match

In [ ]:
def inspect_files(raw_paths, output_path="files_inspect.csv"):
    ## inspects a list of raw data files and writes a summary csv (columns, row count, step-type values, and basic stats) that can be reviewed manually before import
    output_path = Path(output_path)

    stat_keywords = {
        "energy": ["energy"],
        "time": ["time"],
        "capacity": ["capacity", "cap."],
        "power": ["power"],
    }
    rows = []
    for p in raw_paths:
        df_raw = load_neware_raw(p)
        if df_raw is None:
            continue

        status_col = next(
            (c for c in df_raw.columns if "status" in c.lower() or "step type" in c.lower()),
            None,
        )
        if status_col:
            status_values = df_raw[status_col].unique().tolist()
            last_status = df_raw[status_col].iloc[-1]
        else:
            status_values = None
            last_status = None

        row = {
            "file": Path(p).name,
            "full_path": p,
            "format": Path(p).suffix.lower(),
            "n_rows": len(df_raw),
            "n_columns": len(df_raw.columns),
            "columns": ", ".join(df_raw.columns.tolist()),
            "status_values": ", ".join(map(str, status_values)) if status_values else None,
            "last_status": last_status,
        }

        for group, keywords in stat_keywords.items():
            matched_cols = [c for c in df_raw.columns if any(k in c.lower() for k in keywords)]
            for col in matched_cols:
                if not pd.api.types.is_numeric_dtype(df_raw[col]):
                    continue
                series = df_raw[col]
                row[f"{col} [min]"] = series.min()
                row[f"{col} [max]"] = series.max()
                row[f"{col} [mean]"] = series.mean()

        ## manually editable column: default is "keep". Change to "exclude" for rows you don't want to use (duplicates with a -2/-3 suffix, aborted runs, etc.), save the csv, then run load_filtered_raw_paths
        row["manual_decision"] = "keep"

        rows.append(row)

    df_inspection = pd.DataFrame(rows)
    df_inspection.to_csv(output_path, index=False)
    print(f"Saved inspection for {len(df_inspection)} file(s) to {output_path}")
    print(f"Open {output_path} (e.g. in Excel), edit 'manual_decision' to 'exclude' for unwanted rows, "
          f"save, and only then restart the kernel and run load_filtered_raw_paths.")
    return df_inspection

In [ ]:
def load_filtered_raw_paths(inspection_path="files_inspect.csv", decision_column="manual_decision",
                             path_column="full_path"):
    ## reads files_inspect.csv AFTER manual review of 'manual_decision', and returns only the full_path values marked as 'keep'
    df_inspection = pd.read_csv(inspection_path)

    if decision_column not in df_inspection.columns:
        raise ValueError(
            f"Column '{decision_column}' not found in {inspection_path}. "
            f"Run inspect_files again (current version) before editing manually."
        )

    decision_normalized = df_inspection[decision_column].astype(str).str.strip().str.lower()
    df_kept = df_inspection[decision_normalized == "keep"]

    filtered_paths = df_kept[path_column].tolist()
    n_excluded = len(df_inspection) - len(filtered_paths)
    print(f"load_filtered_raw_paths: {n_excluded} file(s) manually excluded, {len(filtered_paths)} remaining")

    return filtered_paths

**Cell pairing.** `build_charge_discharge_pairs` is the core of the pipeline described in
Methods 3.2: it groups the raw, step-level data into charge/discharge pairs directly from
the recorded electrochemical steps, ignoring rest periods and the Neware `Cycle Index`
entirely. This was necessary because, in half-cells, the Neware `Cycle Index` did not
reliably separate the formation cycle from the first cycle of the main loop (both were
often grouped under the same index); full-cells did not show this problem, but the same
pairing logic is applied to both cell types for consistency. ICE is calculated from these reconstructed pairs, not from the raw Neware cycle numbering.

In [ ]:
def build_charge_discharge_pairs(df):
    ## builds the sequence of charge/discharge pairs from a harmonized dataframe, ignoring 'rest' steps and any step_type that is not a recognized charge/discharge variant. Returns a list of dicts, one per pair, in chronological order. pairs[0] is always the first electrochemical event (i.e. the pair ICE is calculated from).
    step_type_shifted = df["step_type"].shift()
    is_new_step = df["step_type"] != step_type_shifted
    step_block_id = is_new_step.cumsum()

    df = df.copy()
    df["step_block_id"] = step_block_id

    has_current = "current" in df.columns
    has_active_capacity = "active_capacity" in df.columns

    agg_dict = {
        "step_type": ("step_type", "first"),
        "max_charge_capacity": ("charge_capacity", "max"),
        "max_discharge_capacity": ("discharge_capacity", "max"),
    }
    if has_current:
        agg_dict["mean_abs_current"] = ("current", lambda s: s.abs().mean())
    if has_active_capacity:
        ## higher-resolution source: cumulative capacity of the active step, without the 1-decimal truncation present in chg./dchg._cap.(mah) in some half-cell CSV exports
        agg_dict["max_active_capacity"] = ("active_capacity", "max")

    step_summary = df.groupby("step_block_id").agg(**agg_dict).reset_index()
    step_summary = step_summary.sort_values("step_block_id")
    non_rest_steps = step_summary[step_summary["step_type"] != "rest"].reset_index(drop=True)

    ## classify pairs by the SUFFIX of step_type (not a fixed list like "cc_chg"), because Neware uses different variants (cc_chg, cccv_chg, cv_chg, ...) depending on the protocol -- a fixed list would silently ignore unrecognized variants
    unknown_step_types = set()
    pairs = []
    i = 0
    pair_index = 1
    while i < len(non_rest_steps) - 1:
        first_row = non_rest_steps.iloc[i]
        second_row = non_rest_steps.iloc[i + 1]

        first_is_discharge = first_row["step_type"].endswith("dchg")
        first_is_charge = (not first_is_discharge) and first_row["step_type"].endswith("chg")
        second_is_discharge = second_row["step_type"].endswith("dchg")
        second_is_charge = (not second_is_discharge) and second_row["step_type"].endswith("chg")

        first_is_active = first_is_discharge or first_is_charge
        second_is_active = second_is_discharge or second_is_charge

        if not first_is_active:
            unknown_step_types.add(first_row["step_type"])
        if not second_is_active:
            unknown_step_types.add(second_row["step_type"])

        if not first_is_active or not second_is_active:
            i = i + 1
            continue

        if first_is_discharge == second_is_discharge:
            ##  two consecutive steps of the same type (rare) >> skip 1 to avoid getting stuck
            i = i + 1
            continue

        ## when active_capacity is available, it replaces max_charge_capacity/max_discharge_capacity as the value source -- each step_block only has ONE active direction (charge or discharge), so max_active_capacity of that step_block already represents the capacity in that direction, at finer resolution
        if first_is_discharge:
            order = "dchg_first"
            if has_active_capacity:
                charge_capacity = second_row["max_active_capacity"]
                discharge_capacity = first_row["max_active_capacity"]
            else:
                charge_capacity = second_row["max_charge_capacity"]
                discharge_capacity = first_row["max_discharge_capacity"]
        else:
            order = "chg_first"
            if has_active_capacity:
                charge_capacity = first_row["max_active_capacity"]
                discharge_capacity = second_row["max_active_capacity"]
            else:
                charge_capacity = first_row["max_charge_capacity"]
                discharge_capacity = second_row["max_discharge_capacity"]

        pair = {
            "pair_index": pair_index,
            "order": order,
            "charge_capacity": charge_capacity,
            "discharge_capacity": discharge_capacity,
        }
        if has_current:
            pair["mean_abs_current"] = (first_row["mean_abs_current"] + second_row["mean_abs_current"]) / 2

        pairs.append(pair)
        pair_index = pair_index + 1
        i = i + 2

    if unknown_step_types:
        print(f"WARNING: step_type not recognized (neither rest nor ending in chg/dchg), ignored: {unknown_step_types}")

    return pairs

In [ ]:
def organize_data(df, filename, protocol_type=None):
    ## harmonizes column names across the different Neware export formats, detects the protocol type, and builds the charge/discharge pair sequence (replaces the old Cycle-Index-based approach entirely -- see markdown note above)
    df.columns = normalize_column_names(df.columns.tolist())

    rename_map = {
        "cycle_index": "cycle",
        "cycle": "cycle",
        "status": "step_type",
        "step_type": "step_type",
        "voltage(v)": "voltage",
        "voltage": "voltage",
        "current(ma)": "current",
        "chg._cap.(mah)": "charge_capacity",
        "charge_capacity(mah)": "charge_capacity",
        "dchg._cap.(mah)": "discharge_capacity",
        "discharge_capacity(mah)": "discharge_capacity",
        ## combined capacity column with finer resolution than chg./dchg._cap.(mah) in some half-cell CSV files, where the latter is truncated to 1 decimal on export (confirmed in IF/ltl_doe cells: only 5 unique values in the whole file)
        "capacity(mah)": "active_capacity",
        "date": "timestamp",
        "timestamp": "timestamp",
    }
    columns_to_rename = {old: new for old, new in rename_map.items() if old in df.columns}
    df = df.rename(columns=columns_to_rename)
    if "step_type" not in df.columns:
        raise ValueError(f"Could not find a step_type/status column after harmonizing. Columns: {list(df.columns)}")

    df["step_type"] = df["step_type"].astype(str).str.strip().str.lower().str.replace(" ", "_")

    ## protocol_type is inferred from the file extension if not given explicitly
    if protocol_type is None:
        ext = Path(filename).suffix.lower()
        if ext in (".xlsx", ".xls"):
            protocol_type = "ratetest"
        elif ext in (".csv", ".ndax", ".nda"):
            protocol_type = "longterm"
        else:
            protocol_type = "unknown"

    ## builds the charge/discharge pair sequence -- pairs[0] is always pair 1 (first electrochemical event = the pair ICE is defined from)
    pairs = build_charge_discharge_pairs(df)
    if not pairs:
        print(f"WARNING: no charge/discharge pairs found in {filename}, ICE could not be detected.")

    return df, protocol_type, pairs

In [ ]:
def check_step_alternation(df, charge_suffix="chg", discharge_suffix="dchg", rest_value="rest"):
    ## re-applies the same step-block collapsing and rest-filtering used in build_charge_discharge_pairs, and checks whether the resulting step_type sequence alternates strictly between charge and discharge from start to end. Used as a sanity check (Methods 3.2): this is the assumption the whole pairing logic depends on.
    step_type_shifted = df["step_type"].shift()
    is_new_step = df["step_type"] != step_type_shifted
    step_block_id = is_new_step.cumsum()

    df_local = df.copy()
    df_local["step_block_id"] = step_block_id

    step_summary = df_local.groupby("step_block_id").agg(step_type=("step_type", "first")).reset_index()
    step_summary = step_summary.sort_values("step_block_id")
    non_rest_steps = step_summary[step_summary["step_type"] != rest_value].reset_index(drop=True)

    previous_is_discharge = None
    break_position = None
    step_type_at_break = None

    for position in range(len(non_rest_steps)):
        current_step_type = non_rest_steps.loc[position, "step_type"]
        is_discharge = current_step_type.endswith(discharge_suffix)
        is_charge = (not is_discharge) and current_step_type.endswith(charge_suffix)
        is_active = is_discharge or is_charge

        if not is_active:
            ## unrecognized step_type, already handled inside build_charge_discharge_pairs -- does not count as a break here
            continue

        if previous_is_discharge is not None:
            if is_discharge == previous_is_discharge:
                break_position = position
                step_type_at_break = current_step_type
                break

        previous_is_discharge = is_discharge

    result = {
        "n_steps_no_rest": len(non_rest_steps),
        "break_found": break_position is not None,
        "break_position": break_position,
        "step_type_at_break": step_type_at_break,
    }
    return result


In [ ]:
def check_step_alternation_all_cells(raw_dict, filename_dict):
    ##runs check_step_alternation for every cell in raw_dict, using the same harmonized dataframe organize_data already produces -- returns only the cells where the alternation broke
    problems = []

    for cell_code, df_raw in raw_dict.items():
        filename = filename_dict[cell_code]
        df_organized, protocol_type, pairs = organize_data(df_raw, filename)
        result = check_step_alternation(df_organized)

        if result["break_found"]:
            problems.append({
                "cell_code": cell_code,
                "break_position": result["break_position"],
                "step_type_at_break": result["step_type_at_break"],
                "n_steps_no_rest": result["n_steps_no_rest"],
            })

    df_problems = pd.DataFrame(problems)
    print(f"check_step_alternation_all_cells: {len(raw_dict)} cell(s) checked, {len(df_problems)} with an alternation break")
    return df_problems

## 3.3 Electrochemical Target Calculation

The function below implements the target calculation described in Methods 3.3. It is
defined here, ahead of the "run the pipeline" step at the end of Section 3.2, because the
import loop calculates ICE for each cell as soon as it is imported, in the same pass that
writes `master_log.csv`. The two sections stay numbered as in the dissertation; only the
order of the code cells reflects the actual dependency.

Cycle identification depends on cell type: half-cells start with a charge step, full-cells
start with a discharge step. ICE is calculated from the first identified pair as
`Q_second / Q_first`, i.e. order-agnostic with respect to charge/discharge, which keeps a
consistent interpretation of "reversible capacity relative to the first-cycle capacity"
across both cell configurations.

In [ ]:
def calculate_targets(df, protocol_type, pairs):
    ## calculates ICE, n_cycles, and inferred_cell_type from the pair sequence built by build_charge_discharge_pairs. ICE is the sole modelling target -- retention/fade_rate were tried early on and dropped (see Future Plans in the dissertation), so they are not calculated here anymore.
    targets = {
        "ICE": None,
        "n_cycles": None,
        "inferred_cell_type": None,
        "warnings": [],
    }

    if pairs:
        ## cell type is inferred data-driven, from the order of the first pair (half-cells discharge first, full-cells charge first), and cross-checked against the tracker's declared Type in build_master_log -- not looked up externally as ground truth
        if pairs[0]["order"] == "dchg_first":
            targets["inferred_cell_type"] = "half"
        else:
            targets["inferred_cell_type"] = "full"

    if not pairs:
        targets["warnings"].append("no charge/discharge pairs found, ICE not calculated")
        return targets

    ## ICE = second-step capacity / first-step capacity of pair 1, order-agnostic -- NOT hardcoded as discharge/charge, because half-cells discharge first and full-cells charge first; a fixed order would give ICE > 100% for one of the two cell types
    ice_pair = pairs[0]
    if ice_pair["order"] == "dchg_first":
        first_capacity = ice_pair["discharge_capacity"]
        second_capacity = ice_pair["charge_capacity"]
    else:
        first_capacity = ice_pair["charge_capacity"]
        second_capacity = ice_pair["discharge_capacity"]

    if first_capacity == 0:
        targets["warnings"].append("first step capacity at pair 1 is zero, ICE not calculated (division by zero)")
    else:
        ice = (second_capacity / first_capacity) * 100
        if second_capacity == 0:
            targets["warnings"].append("ICE=0%, second step capacity is zero, check cell")
        if ice < 0 or ice > 100:
            targets["warnings"].append(f"ICE={ice:.1f}% outside plausible range (0-100)")
        targets["ICE"] = ice

    ## n_cycles is only meaningful for the longterm protocol ->> rate-test varies C-rate across cycles, so "number of cycles" doesn't mean the same thing there. Used later by flag_low_cycle_count (Section 3.6) to flag cells that barely cycled.
    if protocol_type == "longterm":
        n_cycles = len(pairs)
        targets["n_cycles"] = n_cycles
        if n_cycles < 5:
            targets["warnings"].append(f"Only {n_cycles} cycles with activity, targets may be unreliable")

    return targets


Back to Section 3.2: `build_master_log` consolidates the tracker parameters, the
calculated targets, and any warnings into one row per cell in `master_log.csv`.

In [ ]:
def build_master_log(cell_code, targets, df_track, id_column="cell_code",
                      extra_metadata=None, path="master_log.csv"):
    ## appends one row per cell to master_log.csv: tracker parameters + calculated targets + warnings. If the cell_code already exists in the file, the row is SKIPPED (not appended again) --> the decision of which raw file to use per cell (e.g. when a cell has a repeated/-2/-3 export) should already have been made earlier, at the inspect_files/manual_decision step, so this function only needs to guarantee one entry per cell_code.
    path = Path(path)

    if path.exists():
        existing = pd.read_csv(path)
        existing_codes = existing["cell_code"].tolist()
    else:
        existing = None
        existing_codes = []

    if cell_code in existing_codes:
        print(f"[{cell_code}] already in {path.name}, SKIPPED (not appended again)")
        return cell_code

    row = {"cell_code": cell_code}

    track_row = df_track[df_track[id_column] == cell_code]
    if track_row.empty:
        print(f"[{cell_code}] WARNING: not found in df_track (column '{id_column}'), design parameters missing.")
        track_note = f"cell_code not found in df_track column '{id_column}'"
    else:
        track_dict = track_row.iloc[0].to_dict()
        track_dict.pop(id_column, None)
        row.update(track_dict)
        track_note = None

        declared_type = track_dict.get("type")
        inferred_type = targets.get("inferred_cell_type")
        if declared_type is not None and inferred_type is not None:
            if str(declared_type).strip().lower() != str(inferred_type).strip().lower():
                track_note = f"type mismatch: tracker says '{declared_type}', data inferred '{inferred_type}'"

    if extra_metadata:
        row.update(extra_metadata)

    for key, value in targets.items():
        if key != "warnings":
            row[key] = value

    warnings_list = list(targets.get("warnings", []))
    if track_note:
        warnings_list.append(track_note)
    row["warnings"] = "; ".join(warnings_list)

    df_row = pd.DataFrame([row])
    combined = pd.concat([existing, df_row], ignore_index=True) if existing is not None else df_row
    combined.to_csv(path, index=False)

    print(f"[{cell_code}] Appended to {path.name}")
    return cell_code

### Running the import pipeline

STEP 1 and STEP 2 only need to run once (or again if new raw cells get added later) --
STEP 1 inspects the raw files, then `files_inspect.csv` gets reviewed by hand
(`manual_decision` column), then STEP 2 runs the full import + target calculation and
appends everything to `master_log.csv`.

After that, `master_log.csv` already has everything. So every time the kernel restarts,
there is no need to redo STEP 1/2 (which needs the file dialogs again) -- just run the
checkpoint cell below (STEP 3) to load `master_log.csv` back into memory and continue
from Section 3.4 onward, or wherever the analysis was interrupted.

In [ ]:
## STEP 1 -- select the tracker file and the raw Neware exports, then inspect them. Un-comment, run, review/edit "manual_decision" in files_inspect.csv, save, restart the kernel, and only then run STEP 2 below.

# print("Select tracker file:")
# doe_paths = select_files()
# df_track = load_cell_track(path=doe_paths[0], id_column="cell_code") if doe_paths else None
# known_codes = df_track["cell_code"].astype(str).tolist() if df_track is not None else None

# print("Select Neware raw file(s):")
# raw_paths = select_files()
# df_inspection = inspect_files(raw_paths, output_path="files_inspect.csv")
# df_inspection

In [ ]:
## STEP 2a -- select tracker file, load the reviewed raw file list, build raw_dict (one dataframe per cell, still in its raw/unorganized form)

print("Select tracker file:")
doe_paths = select_files()
df_track = load_cell_track(path=doe_paths[0], id_column="cell_code") if doe_paths else None
known_codes = df_track["cell_code"].astype(str).tolist() if df_track is not None else None

raw_paths_filtered = load_filtered_raw_paths(inspection_path="files_inspect.csv")

raw_dict = {}
filename_dict = {}
if raw_paths_filtered and known_codes:
    for p in raw_paths_filtered:
        df_raw = load_neware_raw(p)
        if df_raw is None:
            continue
        code_value = extract_cell_code(p, known_codes)
        if code_value is None:
            code_value = Path(p).stem
        raw_dict[code_value] = df_raw
        filename_dict[code_value] = Path(p).name
    print(f"raw_dict built with {len(raw_dict)} cell(s).")


In [ ]:
## STEP 2b -- for each cell in raw_dict: organize the raw data, calculate ICE/n_cycles, and append the result to master_log.csv

if raw_dict:
    for cell_code, df_raw in raw_dict.items():
        filename = filename_dict[cell_code]
        df_cell, protocol_type, pairs = organize_data(df_raw, filename)
        targets = calculate_targets(df_cell, protocol_type, pairs)
        cell_code = build_master_log(
            cell_code, targets, df_track,
            id_column="cell_code",
            extra_metadata={"protocol_type": protocol_type, "source_file": filename},
            path="master_log.csv",
        )
    print("Pipeline finished. Check master_log.csv")


**Checks (Methods 3.2).** The pairing logic assumes that active electrochemical steps
alternate strictly between charge and discharge once rest periods are removed. This is
checked below for every imported cell, and the inferred cell type (from the first
identified pair) is cross-checked against the cell type declared in the tracker.

This only works right after STEP 1/2 just ran, since it needs `raw_dict`/`filename_dict` in memory -- not after loading from the STEP 3 checkpoint.

In [ ]:
## check 1: charge/discharge alternation per cell
alternation_problems = check_step_alternation_all_cells(raw_dict, filename_dict)
print(alternation_problems)

## check 2: inferred_cell_type vs. the tracker's declared Type (uses the warning build_master_log already writes when the two disagree)
master_log_check = pd.read_csv("master_log.csv")
type_mismatches = master_log_check[master_log_check["warnings"].str.contains("type mismatch", na=False)]
print(type_mismatches[["cell_code", "inferred_cell_type", "warnings"]])

### Checkpoint -- restart from here

Only needed if STEP 1/2 just ran (or ran in a previous session and the kernel is being
restarted now). `master_log.csv` already has every cell that was imported so far, so this
is the actual starting point most of the time.

In [ ]:
## STEP 3 -- reload master_log.csv from disk (skips STEP 1/2 entirely)
master_log = pd.read_csv("master_log.csv")
print(f"master_log loaded: {len(master_log)} cells")


## Building the analysis-ready dataset

Before any statistical analysis, exploratory analysis, or modelling (Sections 3.4–3.8),
`master_log.csv` is turned into a compact matrix of the columns actually needed
(`master_matrix`), and split into the working groups used throughout the rest of the
notebook: `DOE`, `EXTRA1`, `EXTRA2`, plus `half`/`full` cell-type slices used in the
half-cell/full-cell check (Section 3.5), and a `global` matrix with everything pooled
together (used only in the exploratory analysis, Section 3.4).

`FeatureTransformer` bundles the fitted encoders/scalers (one-hot encoding for the
categorical variables, min-max scaling for the numeric ones, standard scaling for the
targets) so the same fitted transformation can be reused on new points, which matters for
the leave-one-out cross-validation in Section 3.6 (fit only on the training fold, applied
to the held-out point).

In [ ]:
def classify_dataset_group(cell_code):
    ## classifies a cell into its dataset group, based on the cell_code prefix
    if cell_code.startswith("P009"):
        return "EXTRA2"
    if cell_code.startswith("ltl_doe"):
        return "DOE"
    if cell_code.startswith("IF"):
        return "EXTRA1"
    return "other"

In [ ]:
def filter_raw_group(df_matrix, dataset_group=None, protocol_type=None, type_value=None):
    ## filters master_matrix (raw, not yet transformed) by metadata. Called with no filter at all, it returns the "global" group, i.e. every cell.
    mask = pd.Series(True, index=df_matrix.index)

    if dataset_group is not None:
        mask = mask & df_matrix["dataset_group"].isin(dataset_group)
    if protocol_type is not None:
        mask = mask & df_matrix["protocol_type"].isin(protocol_type)
    if type_value is not None:
        mask = mask & df_matrix["type"].isin(type_value)

    df_filtered = df_matrix[mask].reset_index(drop=True)
    print(f"filter_raw_group: {len(df_filtered)} cells kept (dataset_group={dataset_group}, "
          f"protocol_type={protocol_type}, type={type_value})")
    return df_filtered

In [ ]:
class FeatureTransformer:
    ##stores the fitted encoders/scalers for reuse on new points (needed for leakage-safe cross-validation in Section 3.6)

    def __init__(self, categorical_columns=["binder", "solvent", "type"],
                 categories=[["CMC", "PVDF"], ["EC:DEC", "EC:DEC:DMC", "EC:DMC"], ["full", "half"]],
                 numeric_columns=["fec", "thickness"],
                 target_columns=["ICE"]):
        self.categorical_columns = categorical_columns
        self.numeric_columns = numeric_columns
        self.target_columns = target_columns

        self.ohe = OneHotEncoder(categories=categories, sparse_output=False, handle_unknown="error")
        self.numeric_scaler = MinMaxScaler()
        self.target_scalers = {target_column: StandardScaler() for target_column in self.target_columns}

        self.ohe_column_names = None
        self.is_fitted = False

    def fit_transform(self, df_matrix):
        feature_columns_all = self.categorical_columns + self.numeric_columns
        n_before = len(df_matrix)
        df_matrix = df_matrix.dropna(subset=feature_columns_all)
        n_after = len(df_matrix)
        if n_after < n_before:
            print(f"fit_transform: {n_before - n_after} row(s) dropped for missing a feature "
                  f"(e.g. a cell with no match in the tracker)")

        categorical_encoded = self.ohe.fit_transform(df_matrix[self.categorical_columns])
        numeric_scaled = self.numeric_scaler.fit_transform(df_matrix[self.numeric_columns])

        self.ohe_column_names = self.ohe.get_feature_names_out(self.categorical_columns).tolist()
        self.is_fitted = True

        X_categorical = pd.DataFrame(categorical_encoded, columns=self.ohe_column_names, index=df_matrix.index)
        X_numeric = pd.DataFrame(numeric_scaled, columns=self.numeric_columns, index=df_matrix.index)
        X = pd.concat([X_categorical, X_numeric], axis=1)

        y = pd.DataFrame(index=df_matrix.index)
        for target_column in self.target_columns:
            target_series = df_matrix[target_column]
            is_not_null = target_series.notna()
            is_in_range = (target_series >= 0) & (target_series <= 100)
            is_valid = is_not_null & is_in_range

            n_out_of_range = (is_not_null & (~is_in_range)).sum()
            if n_out_of_range > 0:
                print(f"fit_transform: {n_out_of_range} value(s) of '{target_column}' outside 0-100, treated as invalid")

            valid_values = target_series[is_valid].to_numpy().reshape(-1, 1)
            self.target_scalers[target_column].fit(valid_values)

            scaled_column = pd.Series(index=df_matrix.index, dtype="float64")
            scaled_column.loc[is_valid] = self.target_scalers[target_column].transform(valid_values).flatten()
            y[target_column] = scaled_column

        return X, y

    def transform(self, df_matrix):
        if not self.is_fitted:
            raise ValueError("FeatureTransformer has not been fitted yet. Call fit_transform first.")

        feature_columns_all = self.categorical_columns + self.numeric_columns
        df_matrix = df_matrix.dropna(subset=feature_columns_all)

        categorical_encoded = self.ohe.transform(df_matrix[self.categorical_columns])
        numeric_scaled = self.numeric_scaler.transform(df_matrix[self.numeric_columns])

        X_categorical = pd.DataFrame(categorical_encoded, columns=self.ohe_column_names, index=df_matrix.index)
        X_numeric = pd.DataFrame(numeric_scaled, columns=self.numeric_columns, index=df_matrix.index)
        return pd.concat([X_categorical, X_numeric], axis=1)

    def inverse_transform_target(self, target_column, values_scaled):
        values_scaled = np.array(values_scaled).reshape(-1, 1)
        return self.target_scalers[target_column].inverse_transform(values_scaled).flatten()

In [ ]:
def build_master_matrix(df_master, feature_columns=["binder", "solvent", "fec", "thickness", "type"],
                         metadata_columns=["cell_code", "dataset_group", "protocol_type", "n_cycles", "whole_plots"],
                         target_columns=["ICE"]):
    ## keeps only the columns needed downstream (metadata + design parameters + targets), preserving NaN as-is -- filtering/cleaning happens in later, more specific steps
    columns_to_keep = metadata_columns + feature_columns + target_columns
    df_matrix = df_master[columns_to_keep].reset_index(drop=True)
    print(f"build_master_matrix: {len(df_matrix)} cells kept (all groups/protocols, NaN preserved)")
    return df_matrix

In [ ]:
master_log["dataset_group"] = master_log["cell_code"].apply(classify_dataset_group)
print(master_log["dataset_group"].value_counts())

master_matrix = build_master_matrix(master_log)

global_matrix = filter_raw_group(master_matrix)
transformer_global = FeatureTransformer()
X_global, y_global = transformer_global.fit_transform(global_matrix)

doe_matrix_raw = filter_raw_group(master_matrix, dataset_group=["DOE"])
transformer_doe = FeatureTransformer()
X_doe, y_doe = transformer_doe.fit_transform(doe_matrix_raw)

extra1_matrix = filter_raw_group(master_matrix, dataset_group=["EXTRA1"])
extra2_matrix = filter_raw_group(master_matrix, dataset_group=["EXTRA2"])
half_matrix = filter_raw_group(master_matrix, type_value=["half"])
full_matrix = filter_raw_group(master_matrix, type_value=["full"])

print(f"doe: {len(doe_matrix_raw)} cells (before the low-cycle-count check in Section 3.6)")
print(f"if: {len(extra1_matrix)} cells")
print(f"EXTRA2: {len(extra2_matrix)} cells")

## 3.4 Statistical and Exploratory Data Analysis

Following Methods 3.4, this section has three parts: design-space coverage, statistical
relationships between the experimental factors and ICE (ANOVA, split-plot model,
correlations, boxplots), and multivariate exploratory analysis (PCA, k-means, Random
Forest, SHAP) plus a direct comparison of overlapping design points between groups.

Before any of that: ICE values outside the physically plausible 0–100% range get excluded
first (`filter_plausible_ice`), same criterion `FeatureTransformer` already applies.
Affected cells stay in the dataset and get reported, not silently dropped.

In [ ]:
def filter_plausible_ice(df_matrix, target_column="ICE", valid_range=(0.0, 100.0), id_column="cell_code"):
    ## sets ICE to missing for rows outside the physically plausible range (0-100%), before any statistical analysis. Same criterion already used in FeatureTransformer.fit_transform and prepare_measurements (Section 3.8) -> keeps a known data error (e.g. IF2007E5, ICE~2496%) from distorting ANOVA/boxplots/correlations.
    is_not_null = df_matrix[target_column].notna()
    is_in_range = (df_matrix[target_column] >= valid_range[0]) & (df_matrix[target_column] <= valid_range[1])
    is_valid = is_not_null & is_in_range

    n_out_of_range = (is_not_null & (~is_in_range)).sum()
    if n_out_of_range > 0:
        excluded_cell_codes = df_matrix.loc[is_not_null & (~is_in_range), id_column].tolist()
        print(f"filter_plausible_ice: {n_out_of_range} cell(s) with {target_column} outside {valid_range}, "
              f"excluded from statistical analysis: {excluded_cell_codes}")

    df_filtered = df_matrix.copy()
    df_filtered.loc[is_not_null & (~is_in_range), target_column] = None
    return df_filtered

In [ ]:
def build_anova_formula(target_column="ice", categorical_columns=["binder", "solvent"],
                         numeric_columns=["fec", "thickness"]):
    ## builds the R-style formula for statsmodels from column lists -- same categorical_columns/numeric_columns pattern used elsewhere (FeatureTransformer, build_ordinal_features), so formula syntax doesn't need to be written by hand
    terms = [f"C({col})" for col in categorical_columns] + list(numeric_columns)
    formula = f"{target_column} ~ " + " + ".join(terms)
    print(f"build_anova_formula: generated formula = '{formula}'")
    return formula

In [ ]:
def run_anova_ice(df_matrix, categorical_columns=["binder", "solvent"], numeric_columns=["fec", "thickness"],
                   target_column="ICE"):
    ## additive main-effects ANOVA (no interaction terms) via statsmodels: OLS + anova_lm, Type II sums of squares. categorical_columns enter as C(column), numeric_columns enter as numeric (thickness is treated as continuous here only for the purpose of this ANOVA -- its discrete/ordinal nature elsewhere in the pipeline, e.g. NumericalDiscreteParameter in BayBE, is unaffected)
    feature_columns_all = categorical_columns + numeric_columns
    df_valid = df_matrix.dropna(subset=feature_columns_all + [target_column]).copy()
    df_valid = df_valid.rename(columns={target_column: "ice"})

    formula = build_anova_formula(target_column="ice", categorical_columns=categorical_columns,
                                   numeric_columns=numeric_columns)
    ols_model = smf.ols(formula=formula, data=df_valid).fit()
    anova_table = anova_lm(ols_model, typ=2)

    print(f"run_anova_ice: ANOVA run with {len(df_valid)} cells")
    print(anova_table)
    return ols_model, anova_table

In [ ]:
def run_split_plot_anova_ice(df_matrix, whole_plot_column="whole_plots",
                              categorical_columns=["binder", "solvent"], numeric_columns=["fec", "thickness"],
                              target_column="ICE"):
    ##split-plot version of run_anova_ice: thickness (whole-plot factor) and binder/solvent/fec (sub-plot factors) enter as fixed effects, but the whole plot itself enters as a random effect (random intercept), via MixedLM. whole_plots is already filled in master_log/tracker (NaN for cells outside the DOE, which don't have this structure).
    feature_columns_all = categorical_columns + numeric_columns + [whole_plot_column]
    df_valid = df_matrix.dropna(subset=feature_columns_all + [target_column]).copy()
    df_valid = df_valid.rename(columns={target_column: "ice"})

    formula = build_anova_formula(target_column="ice", categorical_columns=categorical_columns,
                                   numeric_columns=numeric_columns)

    mixed_model = smf.mixedlm(formula, data=df_valid, groups=df_valid[whole_plot_column])
    result = mixed_model.fit()

    print(f"run_split_plot_anova_ice: mixed model run with {len(df_valid)} cells in "
          f"{df_valid[whole_plot_column].nunique()} whole plot(s)")
    print(result.summary())
    return result

In [ ]:
def build_design_coverage_table(df_matrix, groupby_columns=["binder", "solvent", "thickness"],
                                 group_column="dataset_group", id_column="cell_code"):
    ## counts how many cells (replicates) exist per design combination (binder x solvent x thickness by default), both overall and per dataset group -- shows where the design space has few or no points
    coverage_total = df_matrix.groupby(groupby_columns, as_index=False).agg(n_cells=(id_column, "count"))

    coverage_by_group = df_matrix.groupby(groupby_columns + [group_column], as_index=False).agg(
        n_cells=(id_column, "count"),
    )
    coverage_by_group_pivot = coverage_by_group.pivot_table(
        index=groupby_columns, columns=group_column, values="n_cells", fill_value=0,
    ).reset_index()

    print(f"build_design_coverage_table: {len(coverage_total)} unique combination(s) of {groupby_columns}")
    return coverage_total, coverage_by_group_pivot

In [ ]:
def build_correlation_matrix(df_matrix, numeric_columns=["fec", "thickness"], target_column="ICE"):
    ## Pearson correlation between fec/thickness and the target, within a single data group (by default, DOE -- the only group with broad coverage of both variables; IF and P009 fix solvent=EC:DEC and thickness=200um, so running the correlation on them together with DOE would confound "group effect" with "solvent/thickness effect")
    rows = []
    for numeric_column in numeric_columns:
        df_pair = df_matrix[[numeric_column, target_column]].dropna()
        n_points = len(df_pair)
        if n_points < 3:
            correlation, p_value = None, None
        else:
            correlation, p_value = pearsonr(df_pair[numeric_column], df_pair[target_column])
        rows.append({"variable": numeric_column, "n_points": n_points, "correlation": correlation, "p_value": p_value})

    return pd.DataFrame(rows)


In [ ]:
def build_correlation_matrix_by_group(df_matrix, numeric_columns=["fec", "thickness"],
                                       target_column="ICE", group_column="dataset_group"):
    ## same correlation as above, but split by dataset group -- flags the type of confound described above (e.g. an apparent solvent effect that in fact only exists within the DOE)
    groups = sorted(df_matrix[group_column].dropna().unique().tolist())

    rows = []
    for group in groups:
        df_group = df_matrix[df_matrix[group_column] == group]
        for numeric_column in numeric_columns:
            df_pair = df_group[[numeric_column, target_column]].dropna()
            n_points = len(df_pair)
            if n_points < 3:
                correlation, p_value = None, None
            else:
                correlation, p_value = pearsonr(df_pair[numeric_column], df_pair[target_column])
            rows.append({
                "group": group, "variable": numeric_column,
                "n_points": n_points, "correlation": correlation, "p_value": p_value,
            })

    return pd.DataFrame(rows)

In [ ]:
def plot_ice_by_group(df_matrix, group_column, target_column="ICE", save_path=None):
    ##boxplot + strip plot (individual points with jitter) of ICE (or another target) by group
    groups = sorted(df_matrix[group_column].dropna().unique().tolist())
    values_per_group = [df_matrix[df_matrix[group_column] == group][target_column].dropna().tolist()
                         for group in groups]

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.boxplot(values_per_group, labels=groups, showfliers=False)

    jitter_rng = np.random.default_rng(42)
    for i, values in enumerate(values_per_group):
        x_jitter = jitter_rng.normal(loc=i + 1, scale=0.05, size=len(values))
        ax.scatter(x_jitter, values, alpha=0.6, s=15, color="black")

    ax.set_xlabel(group_column)
    ax.set_ylabel(target_column)
    ax.set_title(f"{target_column} by {group_column}")
    ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=120)
        print(f"Saved plot to {save_path}")
    plt.show()

In [ ]:
def find_matching_design_points(df_matrix, params=["binder", "solvent", "thickness", "fec"],
                                 group_column="dataset_group", id_column="cell_code",
                                 fec_round_decimals=3, thickness_round_decimals=0):
    ## finds exact design combinations (params) that appear in more than one dataset group -- known overlaps: DOE x IF (1 combination) and half (IF, binder=CMC) x P009 (5 combinations, binder/solvent/thickness fixed, fec varies). fec/thickness are rounded before comparing for exact equality, same reason as the rounding in prepare_measurements (Section 3.8): floating-point precision differs slightly between sources.
    df_matrix = df_matrix.copy()
    df_matrix["fec"] = df_matrix["fec"].round(fec_round_decimals)
    df_matrix["thickness"] = df_matrix["thickness"].round(thickness_round_decimals)

    group_count_per_combo = df_matrix.groupby(params)[group_column].nunique().reset_index()
    group_count_per_combo = group_count_per_combo.rename(columns={group_column: "n_groups"})

    overlapping_combos = group_count_per_combo[group_count_per_combo["n_groups"] > 1]
    if overlapping_combos.empty:
        print("find_matching_design_points: no overlapping design combination between groups")
        return pd.DataFrame(), overlapping_combos

    df_merge = df_matrix.merge(overlapping_combos[params], on=params, how="inner")

    columns_to_keep = params + [group_column, id_column, "ICE"]
    columns_present = [col for col in columns_to_keep if col in df_merge.columns]
    df_overlap = df_merge[columns_present].sort_values(params + [group_column]).reset_index(drop=True)

    print(f"find_matching_design_points: {len(overlapping_combos)} overlapping design combination(s) "
          f"between groups, {len(df_overlap)} cell(s) involved")
    return df_overlap, overlapping_combos

In [ ]:
def plot_matching_design_points(df_overlap, params=["binder", "solvent", "thickness", "fec"],
                                 group_column="dataset_group", target_column="ICE", save_path=None):
    ## plots ICE by dataset group, one cluster of bars per overlapping design combination
    if df_overlap.empty:
        print("plot_matching_design_points: nothing to plot")
        return

    df_overlap = df_overlap.copy()
    combo_labels = []
    for i in range(len(df_overlap)):
        row = df_overlap.iloc[i]
        combo_labels.append(", ".join(f"{param}={row[param]}" for param in params))
    df_overlap["combo_label"] = combo_labels

    combos = sorted(df_overlap["combo_label"].unique().tolist())
    groups = sorted(df_overlap[group_column].dropna().unique().tolist())

    fig, ax = plt.subplots(figsize=(10, 6))
    bar_width = 0.8 / len(groups)

    for group_i, group in enumerate(groups):
        means_per_combo = []
        for combo in combos:
            df_filter = df_overlap[(df_overlap["combo_label"] == combo) & (df_overlap[group_column] == group)]
            means_per_combo.append(df_filter[target_column].mean())

        x_positions = [combo_i + group_i * bar_width for combo_i in range(len(combos))]
        ax.bar(x_positions, means_per_combo, width=bar_width, label=group)

    x_tick_positions = [combo_i + (len(groups) - 1) * bar_width / 2 for combo_i in range(len(combos))]
    ax.set_xticks(x_tick_positions)
    ax.set_xticklabels(combos, rotation=45, ha="right")

    ax.set_ylabel(target_column)
    ax.set_title(f"{target_column} at overlapping design points, by group")
    ax.legend()
    ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=120)
        print(f"Saved plot to {save_path}")
    plt.show()

### Running the statistical analysis

`master_matrix_stats` applies `filter_plausible_ice` to the full dataset. The split-plot
model and the DOE-only ANOVA/correlations/boxplots use `doe_matrix_stats` (the only group
with a proper factorial design); the coverage table, the full-dataset ANOVA, and the
overlapping-design-point comparison use the full `master_matrix_stats`.

In [ ]:
master_matrix_stats = filter_plausible_ice(master_matrix)
doe_matrix_stats = master_matrix_stats[master_matrix_stats["dataset_group"] == "DOE"]

print(f"master_matrix_stats: {len(master_matrix_stats)} cells, {master_matrix_stats['ICE'].isna().sum()} with missing/excluded ICE")
print(f"doe_matrix_stats: {len(doe_matrix_stats)} cells")

In [ ]:
## split-plot mixed-effects model -- DOE only (Table S3)
split_plot_result_doe = run_split_plot_anova_ice(doe_matrix_stats)

In [ ]:
## design-space coverage (binder x solvent x thickness), overall and by group
coverage_total, coverage_by_group = build_design_coverage_table(master_matrix_stats)
print(coverage_total)
print(coverage_by_group)

In [ ]:
## Pearson correlations (fec, thickness) x ICE -- DOE only, and split by group
correlation_doe = build_correlation_matrix(doe_matrix_stats)
print(correlation_doe)

correlation_by_group = build_correlation_matrix_by_group(master_matrix_stats)
print(correlation_by_group)

In [ ]:
## ANOVA -- full dataset, with dataset_group added as a factor (Table S5)
anova_model_full, anova_table_full = run_anova_ice(
    master_matrix_stats,
    categorical_columns=["binder", "solvent", "dataset_group"],
    numeric_columns=["fec", "thickness"],
)

In [ ]:
## ANOVA -- DOE only (Table S4). The DOE is the only group that varies all four factors freely, without the design confounds present in IF/P009 (which fix solvent=EC:DEC and thickness=200um)
anova_model_doe, anova_table_doe = run_anova_ice(doe_matrix_stats)

In [ ]:
## boxplots -- DOE only (Figures S3-S6)
plot_ice_by_group(doe_matrix_stats, "binder")
plot_ice_by_group(doe_matrix_stats, "solvent")
plot_ice_by_group(doe_matrix_stats, "thickness")
plot_ice_by_group(doe_matrix_stats, "fec")

## boxplot -- ICE by dataset group, full dataset (Figure S7)
plot_ice_by_group(master_matrix_stats, "dataset_group")

**Overlapping design points.** `find_matching_design_points` is run on the raw
`master_matrix` (not the ICE-filtered version), so the known IF2007E5 data error still
shows up in the overlap table/plot as context, rather than being hidden.

In [ ]:
df_overlap, overlapping_combos = find_matching_design_points(master_matrix_stats)
print(df_overlap)

plot_matching_design_points(df_overlap, target_column="ICE")


### Multivariate exploratory analysis

PCA and k-means are applied to the experimental input variables only (ICE excluded from
the feature representation). Random Forest feature importance and SHAP are used as
exploratory, non-predictive tools to look for associations between the input variables
and ICE -- the model is evaluated on the same data it was fitted on, so these results are
interpreted as exploratory associations, not predictive performance (Methods 3.4).

In [ ]:
def run_pca(X, n_components=2):
    pca = PCA(n_components=n_components)
    scores = pca.fit_transform(X)

    column_names = [f"pc{i + 1}" for i in range(n_components)]
    df_scores = pd.DataFrame(scores, columns=column_names, index=X.index)

    variance_explained = pca.explained_variance_ratio_
    for i in range(n_components):
        print(f"PC{i + 1} explains {variance_explained[i] * 100:.1f}% of the variance")

    return pca, df_scores


In [ ]:
def get_group_metadata(df_matrix, index_reference, metadata_columns=["cell_code", "dataset_group", "protocol_type", "type"]):
    return df_matrix.loc[index_reference, metadata_columns]


In [ ]:
def plot_pca_scores(df_scores, df_metadata, group_column, save_path=None):
    fig, ax = plt.subplots(figsize=(8, 6))

    for group in sorted(df_metadata[group_column].dropna().unique().tolist()):
        mask = df_metadata[group_column] == group
        ax.scatter(df_scores[mask]["pc1"], df_scores[mask]["pc2"], alpha=0.7, label=str(group))

    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title(f"PCA scores coloured by {group_column}")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=120)
        print(f"Saved plot to {save_path}")
    plt.show()

In [ ]:
def run_kmeans_elbow(X, k_range=[2, 3, 4, 5, 6, 7, 8]):
    inertia_values = []
    for k in k_range:
        kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
        kmeans.fit(X)
        inertia_values.append(kmeans.inertia_)
    return pd.DataFrame({"k": k_range, "inertia": inertia_values})


In [ ]:
def plot_kmeans_elbow(df_elbow, save_path=None):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(df_elbow["k"], df_elbow["inertia"], marker="o")
    ax.set_xlabel("Number of clusters (k)")
    ax.set_ylabel("Inertia")
    ax.set_title("Elbow method")
    ax.grid(alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=120)
        print(f"Saved plot to {save_path}")
    plt.show()


In [ ]:
def run_kmeans(X, n_clusters=3):
    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    labels = kmeans.fit_predict(X)
    cluster_labels = pd.Series(labels, index=X.index, name="cluster")

    print(f"run_kmeans: {n_clusters} clusters, count per cluster:")
    print(cluster_labels.value_counts().sort_index())
    return kmeans, cluster_labels

In [ ]:
def run_feature_importance(X, y_target, n_estimators=200):
    is_valid = y_target.notna()
    X_valid, y_valid = X[is_valid], y_target[is_valid]

    rf = RandomForestRegressor(n_estimators=n_estimators, random_state=42)
    rf.fit(X_valid, y_valid)

    importance_df = pd.DataFrame({"feature": X.columns.tolist(), "importance": rf.feature_importances_})
    importance_df = importance_df.sort_values("importance", ascending=False).reset_index(drop=True)

    ## this R2 is evaluated on the training data (rf.score on X_valid/y_valid, the same rows used to fit rf), so it is a training R2, not a held-out/predictive one. Reported here only as a training-fit diagnostic -- exploratory associations and feature importance are the actual result used in Section 4.1, not this score.
    train_r_squared = rf.score(X_valid, y_valid)
    print(f"run_feature_importance: training R2 = {train_r_squared:.3f} (n={len(X_valid)} cells)")

    return rf, importance_df


In [ ]:
def plot_feature_importance(importance_df, save_path=None):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(importance_df["feature"], importance_df["importance"])
    ax.set_xlabel("Importance")
    ax.set_title("Feature importance (Random Forest)")
    ax.invert_yaxis()
    ax.grid(alpha=0.3, axis="x")
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=120)
        print(f"Saved plot to {save_path}")
    plt.show()

In [ ]:
def run_shap(rf, X, y_target):
    is_valid = y_target.notna()
    X_valid = X[is_valid]

    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_valid)
    return explainer, shap_values, X_valid


In [ ]:
def plot_shap_summary(shap_values, X_valid, save_path=None):
    shap.summary_plot(shap_values, X_valid, show=False)
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
        print(f"Saved plot to {save_path}")
    plt.show()

Run on two scopes: `global` (every cell pooled together) and `DOE` only. Per SI-4,
Random Forest and SHAP results on the full dataset should be read with the caveat that FEC
appears as the dominant factor there mainly because IF and P009 vary almost exclusively
FEC -- not an independent finding.

In [ ]:
## global scope (all groups pooled)
pca_global, scores_global = run_pca(X_global)
metadata_global = get_group_metadata(global_matrix, X_global.index)
plot_pca_scores(scores_global, metadata_global, "dataset_group")
plot_pca_scores(scores_global, metadata_global, "type")

elbow_global = run_kmeans_elbow(X_global)
plot_kmeans_elbow(elbow_global)
kmeans_global, clusters_global = run_kmeans(X_global, n_clusters=3)

rf_global, importance_global = run_feature_importance(X_global, y_global["ICE"])
plot_feature_importance(importance_global)

explainer_global, shap_values_global, X_valid_global = run_shap(rf_global, X_global, y_global["ICE"])
plot_shap_summary(shap_values_global, X_valid_global)

In [ ]:
## DOE-only scope
pca_doe, scores_doe = run_pca(X_doe)
metadata_doe = get_group_metadata(doe_matrix_raw, X_doe.index)
plot_pca_scores(scores_doe, metadata_doe, "type")

elbow_doe = run_kmeans_elbow(X_doe)
plot_kmeans_elbow(elbow_doe)
kmeans_doe, clusters_doe = run_kmeans(X_doe, n_clusters=3)

rf_doe, importance_doe = run_feature_importance(X_doe, y_doe["ICE"])
plot_feature_importance(importance_doe)

explainer_doe, shap_values_doe, X_valid_doe = run_shap(rf_doe, X_doe, y_doe["ICE"])
plot_shap_summary(shap_values_doe, X_valid_doe)

## 3.5 Half-Cell/Full-Cell Trend Consistency Check

Section 3.4 shows where the dataset groups overlap in the design space. This section asks
a more specific question: within the region shared by half-cells and full-cells, do the
predicted ICE trends agree between the two configurations? RQ3.

In [ ]:
def build_ordinal_features(df_matrix, categorical_columns=["binder", "solvent", "type"],
                            categories=[["CMC", "PVDF"], ["EC:DEC", "EC:DEC:DMC", "EC:DMC"], ["full", "half"]],
                            numeric_columns=["fec", "thickness"],
                            numeric_bounds={"thickness": (100.0, 750.0)},
                            target_columns=["ICE"]):
    ## ordinal-encodes the categorical columns and min-max-scales the numeric columns, for use with BoTorch's MixedSingleTaskGP (which expects a Hamming-kernel-ready ordinal encoding for categorical dims, not one-hot). Numeric columns are scaled using the REAL PHYSICAL BOUNDS of what the lab can produce (e.g. thickness from 100 to 750), passed via numeric_bounds -- not just the min/max of what has already been tested. Columns without an explicit bound in numeric_bounds use the observed min/max instead.
    feature_columns_all = categorical_columns + numeric_columns
    df_matrix = df_matrix.dropna(subset=feature_columns_all).copy()

    ordinal_encoder = OrdinalEncoder(categories=categories)
    categorical_encoded = ordinal_encoder.fit_transform(df_matrix[categorical_columns])
    X_categorical = pd.DataFrame(categorical_encoded, columns=categorical_columns, index=df_matrix.index)

    X_numeric = pd.DataFrame(index=df_matrix.index)
    numeric_bounds_used = {}
    for numeric_column in numeric_columns:
        if numeric_column in numeric_bounds:
            value_min, value_max = numeric_bounds[numeric_column]
        else:
            value_min, value_max = df_matrix[numeric_column].min(), df_matrix[numeric_column].max()
        numeric_bounds_used[numeric_column] = (value_min, value_max)
        X_numeric[numeric_column] = (df_matrix[numeric_column] - value_min) / (value_max - value_min)

    X = pd.concat([X_categorical, X_numeric], axis=1)

    y = pd.DataFrame(index=df_matrix.index)
    target_scalers = {}
    for target_column in target_columns:
        target_series = df_matrix[target_column]
        is_not_null = target_series.notna()
        is_in_range = (target_series >= 0) & (target_series <= 100)
        is_valid = is_not_null & is_in_range

        valid_values = target_series[is_valid].to_numpy().reshape(-1, 1)
        target_scaler = StandardScaler()
        target_scaler.fit(valid_values)
        target_scalers[target_column] = target_scaler

        scaled_column = pd.Series(index=df_matrix.index, dtype="float64")
        scaled_column.loc[is_valid] = target_scaler.transform(valid_values).flatten()
        y[target_column] = scaled_column

    return X, y, ordinal_encoder, numeric_bounds_used, target_scalers

`build_ordinal_features` gives an ordinal (not one-hot) encoding, because
`build_single_or_mixed_gp` below needs to know which columns are categorical to point a
`MixedSingleTaskGP` at them (falling back to a plain `SingleTaskGP` when nothing
categorical has variation left -- e.g. the full-cell group, where binder/solvent/thickness
are all fixed by design and get dropped entirely).

In [ ]:
def build_single_or_mixed_gp(X, y_column, categorical_column_names=["binder", "solvent", "type"]):
    is_valid = y_column.notna()
    X_valid = X[is_valid].copy()
    y_valid = y_column[is_valid]

    ##  columns with no variation left (e.g. binder/solvent in the full-cell group, where they are fixed by design) are dropped -- BoTorch's GPs need at least some variance per input dimension
    constant_columns = [col for col in X_valid.columns if X_valid[col].nunique() <= 1]
    if constant_columns:
        print(f"build_single_or_mixed_gp: dropping columns with no variation: {constant_columns}")
        X_valid = X_valid.drop(columns=constant_columns)

    feature_names_used = X_valid.columns.tolist()
    cat_dims = [feature_names_used.index(name) for name in categorical_column_names if name in feature_names_used]

    X_tensor = torch.tensor(X_valid.to_numpy())
    y_tensor = torch.tensor(y_valid.to_numpy()).unsqueeze(-1)

    ## MixedSingleTaskGP requires at least 1 categorical dimension -- if every categorical column was dropped for lack of variance (e.g. full/P009, where binder and solvent are fixed), the problem is no longer "mixed" and becomes purely continuous. In that case, fall back to a plain SingleTaskGP.
    if len(cat_dims) == 0:
        print("build_single_or_mixed_gp: no categorical column with remaining variation, using SingleTaskGP")
        model = SingleTaskGP(X_tensor, y_tensor)
    else:
        model = MixedSingleTaskGP(X_tensor, y_tensor, cat_dims=cat_dims)

    mll = ExactMarginalLogLikelihood(model.likelihood, model)
    fit_gpytorch_mll(mll, optimizer_kwargs={"options": {"maxiter": 2000}})

    print(f"build_single_or_mixed_gp: model trained with {len(X_valid)} cells, {len(feature_names_used)} features")
    return model, feature_names_used, cat_dims

The next three helpers build the restricted, FEC-only comparison grid: since the full-cell
group (`EXTRA2`) fixes binder/solvent/thickness and only varies FEC, the comparison has to
happen on that same restricted slice, or the two groups wouldn't be predicting on the same
region of the design space.

In [ ]:
def get_fixed_values_from_full(full_matrix, expected_fixed_columns=["binder", "solvent", "thickness"]):
    ## reads off the fixed values of the full-cell group directly from the data, instead of hardcoding them -- works for any full_matrix, not just EXTRA2 specifically. If a column is not actually constant (nunique > 1), this warns explicitly instead of silently picking an arbitrary value.
    fixed_values = {}
    for column in expected_fixed_columns:
        unique_values = full_matrix[column].dropna().unique()
        if len(unique_values) == 0:
            raise ValueError(f"get_fixed_values_from_full: column '{column}' has no valid value in full_matrix")
        if len(unique_values) > 1:
            print(f"get_fixed_values_from_full: WARNING -- column '{column}' is NOT constant in full_matrix "
                  f"({unique_values}), using the most frequent value")
            chosen_value = full_matrix[column].mode().iloc[0]
        else:
            chosen_value = unique_values[0]
        fixed_values[column] = chosen_value

    print(f"get_fixed_values_from_full: detected fixed values = {fixed_values}")
    return fixed_values

In [ ]:
def build_fec_only_grid(fixed_binder, fixed_solvent, fixed_thickness,
                         fec_values_raw=[0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]):
    ## restricted evaluation grid: binder/solvent/thickness fixed (from get_fixed_values_from_full), only fec varies
    rows = [{"binder": fixed_binder, "solvent": fixed_solvent, "thickness": fixed_thickness, "fec": fec}
            for fec in fec_values_raw]
    return pd.DataFrame(rows)

In [ ]:
def predict_grid_with_mixed_gp(model, df_grid, ordinal_encoder, numeric_bounds_used, feature_names_used,
                                categorical_columns=["binder", "solvent"], numeric_columns=["fec", "thickness"]):
    ##  feature_names_used: the columns actually used to train the model (after build_single_or_mixed_gp dropped columns with no variance, e.g. binder/solvent/thickness for the full-cell group) -- the evaluation grid MUST be reduced/reordered to the same columns, or the tensor passed to model.posterior() has a different dimension from the training tensor and errors out
    categorical_encoded = ordinal_encoder.transform(df_grid[categorical_columns])
    X_categorical = pd.DataFrame(categorical_encoded, columns=categorical_columns, index=df_grid.index)

    X_numeric = pd.DataFrame(index=df_grid.index)
    for numeric_column in numeric_columns:
        value_min, value_max = numeric_bounds_used[numeric_column]
        X_numeric[numeric_column] = (df_grid[numeric_column] - value_min) / (value_max - value_min)

    X_grid = pd.concat([X_categorical, X_numeric], axis=1)[feature_names_used]
    X_grid_tensor = torch.tensor(X_grid.to_numpy())

    model.eval()
    with torch.no_grad():
        mean_scaled = model.posterior(X_grid_tensor).mean.detach().numpy().flatten()
    return mean_scaled

`check_half_full_convergence` puts all of the above together: fit one GP on the half-cell
data and one on the full-cell data, predict both on the same FEC-only grid, then compare
the two predicted trends with Spearman's rank correlation. The result is un-scaled back to
real ICE (%) only so the output table is readable -- Spearman's rho is invariant to that
linear rescaling, so the correlation itself doesn't change.

In [ ]:
def check_half_full_convergence(half_matrix, full_matrix,
                                 fec_values_raw=[0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]):
    fixed_values = get_fixed_values_from_full(full_matrix)

    df_grid = build_fec_only_grid(
        fixed_binder=fixed_values["binder"], fixed_solvent=fixed_values["solvent"],
        fixed_thickness=fixed_values["thickness"], fec_values_raw=fec_values_raw,
    )

    X_half_ord, y_half_ord, encoder_half, bounds_half, scalers_half = build_ordinal_features(
        half_matrix, categorical_columns=["binder", "solvent"],
        categories=[["CMC", "PVDF"], ["EC:DEC", "EC:DEC:DMC", "EC:DMC"]],
        numeric_columns=["fec", "thickness"], target_columns=["ICE"],
    )
    X_full_ord, y_full_ord, encoder_full, bounds_full, scalers_full = build_ordinal_features(
        full_matrix, categorical_columns=["binder", "solvent"],
        categories=[["CMC", "PVDF"], ["EC:DEC", "EC:DEC:DMC", "EC:DMC"]],
        numeric_columns=["fec", "thickness"], target_columns=["ICE"],
    )

    model_half, features_half, cat_dims_half = build_single_or_mixed_gp(
        X_half_ord, y_half_ord["ICE"], categorical_column_names=["binder", "solvent"])
    model_full, features_full, cat_dims_full = build_single_or_mixed_gp(
        X_full_ord, y_full_ord["ICE"], categorical_column_names=["binder", "solvent"])

    print(f"check_half_full_convergence: half-cell model uses features {features_half}")
    print(f"check_half_full_convergence: full-cell model uses features {features_full}")

    predictions_half_scaled = predict_grid_with_mixed_gp(model_half, df_grid, encoder_half, bounds_half, features_half)
    predictions_full_scaled = predict_grid_with_mixed_gp(model_full, df_grid, encoder_full, bounds_full, features_full)

    ## un-does the StandardScaler to report ICE in real units (%), not scaled -- Spearman rho is invariant to this linear transform, so the correlation result doesn't change; this is only to make the table readable/citable
    predictions_half = scalers_half["ICE"].inverse_transform(predictions_half_scaled.reshape(-1, 1)).flatten()
    predictions_full = scalers_full["ICE"].inverse_transform(predictions_full_scaled.reshape(-1, 1)).flatten()

    correlation, p_value = spearmanr(predictions_half, predictions_full)
    print(f"check_half_full_convergence: Spearman rho = {correlation:.3f} (p={p_value:.4f}), "
          f"n_grid_points={len(df_grid)}")
    print(f"check_half_full_convergence: grid restricted to binder={fixed_values['binder']}, "
          f"solvent={fixed_values['solvent']}, thickness={fixed_values['thickness']}, "
          f"fec varying over {fec_values_raw}")

    df_grid_result = df_grid.copy()
    df_grid_result["ICE_predicted_half"] = predictions_half
    df_grid_result["ICE_predicted_full"] = predictions_full
    return df_grid_result, correlation, p_value

Two versions of this check are reported in the dissertation (Section 4.4): the full EXTRA1
group against `EXTRA2`, and a more tightly controlled version restricting the half-cell data
group filtered to `binder == "CMC"` -- so both groups being compared share the same degree
of design restriction (only FEC varies in both). Only the second, more controlled version
is run here, since it is the one reported as the main GP-based comparison.

In [ ]:
extra1_cmc_matrix = extra1_matrix[extra1_matrix["binder"] == "CMC"].reset_index(drop=True)
print(f"IF binder=CMC: {len(extra1_cmc_matrix)} cells (of {len(extra1_matrix)} in IF total)")

convergence_result_extra1_cmc, rho_extra1_cmc, p_value_extra1_cmc = check_half_full_convergence(extra1_cmc_matrix, full_matrix)
print(convergence_result_extra1_cmc)

## 3.6 Bayesian Optimisation Framework -- Kernel Ablation and LOO-CV

Phase 1 (BoTorch, DOE-only dataset), addressing RQ1: does explicitly modelling
categorical variables (binder, solvent) improve surrogate predictive performance compared
to a standard one-hot-encoded baseline? Three GP configurations are compared by
leave-one-out cross-validation (Table 2): OHE+RBF, OHE+Matérn, and Hamming+Matérn.

In [ ]:
def build_rbf_single_task_gp(X_tensor, y_tensor, ard_num_dims):
    ## trains a SingleTaskGP with an RBF (ARD) kernel from ready-made tensors -- used for the OHE+RBF baseline configuration
    base_kernel = RBFKernel(ard_num_dims=ard_num_dims, lengthscale_prior=GammaPrior(3.0, 6.0))
    covar_module = ScaleKernel(base_kernel, outputscale_prior=GammaPrior(2.0, 0.15))
    mean_module = ConstantMean()

    model = SingleTaskGP(X_tensor, y_tensor, mean_module=mean_module, covar_module=covar_module)
    mll = ExactMarginalLogLikelihood(model.likelihood, model)
    fit_gpytorch_mll(mll, optimizer_kwargs={"options": {"maxiter": 2000}})
    return model


In [ ]:
def build_matern_single_task_gp(X_tensor, y_tensor, ard_num_dims):
    ## same as build_rbf_single_task_gp, but with a Matern (nu=2.5) kernel instead of RBF -- used for the OHE+Matern configuration, to isolate the effect of the kernel family from the effect of the categorical encoding
    base_kernel = MaternKernel(nu=2.5, ard_num_dims=ard_num_dims, lengthscale_prior=GammaPrior(3.0, 6.0))
    covar_module = ScaleKernel(base_kernel, outputscale_prior=GammaPrior(2.0, 0.15))
    mean_module = ConstantMean()

    model = SingleTaskGP(X_tensor, y_tensor, mean_module=mean_module, covar_module=covar_module)
    mll = ExactMarginalLogLikelihood(model.likelihood, model)
    fit_gpytorch_mll(mll, optimizer_kwargs={"options": {"maxiter": 2000}})
    return model

These two builders give us the first two configurations: **OHE+RBF** (the baseline -- 
`build_rbf_single_task_gp`) and **OHE+Matérn** (same one-hot encoding, only the kernel
family changes -- `build_matern_single_task_gp`). Comparing these two in isolation is what
lets us tell apart "the kernel family matters" from "the categorical encoding matters"
in Section 4.2, since the third configuration below changes both at once.

Both keep the kernel priors (Gamma(3.0, 6.0) on lengthscale, Gamma(2.0, 0.15) on
outputscale) at BoTorch's own defaults for `SingleTaskGP`, rather than tuning them
manually -- with only 38 DOE observations there's no reliable way to tune priors
independently of the data used to evaluate the model.

In [ ]:
def plot_loo_cv(df_result, target_name="target", save_path=None):
    r2 = r2_score(df_result["actual"], df_result["predicted"])
    rmse = mean_squared_error(df_result["actual"], df_result["predicted"]) ** 0.5

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(df_result["actual"], df_result["predicted"], alpha=0.7)

    value_min = min(df_result["actual"].min(), df_result["predicted"].min())
    value_max = max(df_result["actual"].max(), df_result["predicted"].max())
    ax.plot([value_min, value_max], [value_min, value_max], linestyle="--", color="gray")

    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted by the GP")
    ax.set_title(f"Leave-one-out CV -- {target_name}\nR2={r2:.3f}, RMSE={rmse:.3f}")
    ax.grid(alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=120)
        print(f"Saved plot to {save_path}")
    plt.show()

    return r2, rmse

**Why leave-one-out, not five-fold.** With only ~40 cells in the DOE, five-fold CV would
leave very little data per fold to fit the GP on. LOO-CV holds out one cell at a time and
fits on the rest, so every cell gets an out-of-sample prediction while training uses as
much data as possible. `run_loo_cv_no_leakage` below refits the feature/target scalers
independently within each fold (only on the training cells), so no information about the
held-out cell ever leaks into its own prediction.

In [ ]:
def run_loo_cv_no_leakage(df_matrix, target_column, categorical_columns=["binder", "solvent", "type"],
                           categories=[["CMC", "PVDF"], ["EC:DEC", "EC:DEC:DMC", "EC:DMC"], ["full", "half"]],
                           numeric_columns=["fec", "thickness"], gp_builder=build_rbf_single_task_gp,
                           config_label="OHE+RBF"):
    ## leave-one-out CV for a one-hot-encoded SingleTaskGP. gp_builder lets this same loop be reused for both the OHE+RBF baseline and the OHE+Matern configuration (see run_loo_cv_no_leakage calls below) -- the only difference between the two is which kernel-builder function is passed in.
    feature_columns_all = categorical_columns + numeric_columns
    df_valid = df_matrix.dropna(subset=feature_columns_all + [target_column]).reset_index(drop=True)
    n_cells = len(df_valid)

    predictions, actuals = [], []
    for i in range(n_cells):
        train_indices = list(range(n_cells))
        train_indices.remove(i)

        df_train = df_valid.iloc[train_indices].reset_index(drop=True)
        df_test = df_valid.iloc[[i]].reset_index(drop=True)

        transformer_fold = FeatureTransformer(
            categorical_columns=categorical_columns, categories=categories,
            numeric_columns=numeric_columns, target_columns=[target_column],
        )
        X_train, y_train = transformer_fold.fit_transform(df_train)
        X_test = transformer_fold.transform(df_test)

        constant_columns = [col for col in X_train.columns if X_train[col].nunique() <= 1]
        if constant_columns:
            X_train = X_train.drop(columns=constant_columns)
            X_test = X_test.drop(columns=constant_columns)

        y_train_values = y_train[target_column]
        is_valid_train = y_train_values.notna()
        if is_valid_train.sum() < 2:
            print(f"run_loo_cv_no_leakage [{config_label}]: fold {i} skipped, fewer than 2 valid training cells")
            continue

        X_train_valid = X_train[is_valid_train]
        y_train_valid = y_train_values[is_valid_train]

        X_train_tensor = torch.tensor(X_train_valid.to_numpy())
        y_train_tensor = torch.tensor(y_train_valid.to_numpy()).unsqueeze(-1)
        X_test_tensor = torch.tensor(X_test.to_numpy())

        ard_num_dims = X_train_valid.shape[1]
        model = gp_builder(X_train_tensor, y_train_tensor, ard_num_dims)

        model.eval()
        with torch.no_grad():
            prediction_scaled = model.posterior(X_test_tensor).mean.item()

        prediction_actual = transformer_fold.inverse_transform_target(target_column, [prediction_scaled])[0]
        actual_value = df_test[target_column].iloc[0]

        predictions.append(prediction_actual)
        actuals.append(actual_value)
        print(f"run_loo_cv_no_leakage [{config_label}]: fold {i + 1}/{n_cells} trained")

    return pd.DataFrame({"actual": actuals, "predicted": predictions})

**Third configuration: Hamming + Matérn (mixed kernel).** Instead of one-hot encoding,
binder and solvent get a Hamming kernel (treats categories as distinct identities, no
artificial ordering or distance between them) and FEC/thickness keep a Matérn kernel -- 
following Ru et al.'s mixed-kernel approach for BO with categorical and continuous
inputs. `run_loo_cv_mixed_no_leakage` mirrors the loop above, but on the ordinal
encoding + `MixedSingleTaskGP` path, since a Hamming-kernel treatment needs an ordinal
(not one-hot) representation of the categorical columns to know which dimensions to treat
that way.

In [ ]:
def run_loo_cv_mixed_no_leakage(df_matrix, target_column, categorical_columns=["binder", "solvent", "type"],
                                 categories=[["CMC", "PVDF"], ["EC:DEC", "EC:DEC:DMC", "EC:DMC"], ["full", "half"]],
                                 numeric_columns=["fec", "thickness"],
                                 numeric_bounds={"thickness": (100.0, 750.0)}):
    ## leave-one-out CV for the mixed-kernel (Hamming + Matern) configuration -- structurally the same loop as run_loo_cv_no_leakage, but using the ordinal encoding + MixedSingleTaskGP path instead, since a Hamming-kernel treatment of categorical variables needs an ordinal (not one-hot) representation
    feature_columns_all = categorical_columns + numeric_columns
    df_valid = df_matrix.dropna(subset=feature_columns_all + [target_column]).reset_index(drop=True)
    n_cells = len(df_valid)

    predictions, actuals = [], []
    for i in range(n_cells):
        train_indices = list(range(n_cells))
        train_indices.remove(i)

        df_train = df_valid.iloc[train_indices].reset_index(drop=True)
        df_test = df_valid.iloc[[i]].reset_index(drop=True)

        X_train, y_train, encoder_fold, bounds_fold, scalers_fold = build_ordinal_features(
            df_train, categorical_columns=categorical_columns, categories=categories,
            numeric_columns=numeric_columns, numeric_bounds=numeric_bounds, target_columns=[target_column],
        )

        y_train_values = y_train[target_column]
        if y_train_values.notna().sum() < 2:
            print(f"run_loo_cv_mixed_no_leakage: fold {i} skipped, fewer than 2 valid training cells")
            continue

        model, feature_names_used, cat_dims = build_single_or_mixed_gp(
            X_train, y_train_values, categorical_column_names=categorical_columns)

        prediction_scaled = predict_grid_with_mixed_gp(
            model, df_test, encoder_fold, bounds_fold, feature_names_used,
            categorical_columns=categorical_columns, numeric_columns=numeric_columns,
        )[0]

        prediction_actual = scalers_fold[target_column].inverse_transform([[prediction_scaled]])[0][0]
        actual_value = df_test[target_column].iloc[0]

        predictions.append(prediction_actual)
        actuals.append(actual_value)
        print(f"run_loo_cv_mixed_no_leakage: fold {i + 1}/{n_cells} trained")

    return pd.DataFrame({"actual": actuals, "predicted": predictions})

In [ ]:
def flag_low_cycle_count(df_matrix, protocol_type_column="protocol_type", n_cycles_column="n_cycles",
                          min_cycles_threshold=10):
    ##flags cells with few cycles under the longterm protocol -- a proxy for incomplete cycling
    df_matrix = df_matrix.copy()
    df_matrix["low_cycle_flag"] = False

    mask_longterm = df_matrix[protocol_type_column] == "longterm"
    mask_low = df_matrix[n_cycles_column] < min_cycles_threshold
    df_matrix.loc[mask_longterm & mask_low, "low_cycle_flag"] = True

    n_flagged = df_matrix["low_cycle_flag"].sum()
    print(f"flag_low_cycle_count: {n_flagged} cell(s) flagged with fewer than {min_cycles_threshold} cycles")
    return df_matrix

### Running the kernel ablation

First, the low-cycle-count filter is applied and checked for a hidden selection effect on
ICE specifically (Section 4.2): if cells with fewer cycles also tended to have
systematically different ICE, the filter could be introducing a selection bias rather than
just removing noise. Then all three kernel configurations are compared by LOO-CV on the
cleaned DOE dataset.

In [ ]:
doe_matrix_flagged = flag_low_cycle_count(doe_matrix_raw, min_cycles_threshold=10)
doe_matrix_clean = doe_matrix_flagged[doe_matrix_flagged["low_cycle_flag"] == False].reset_index(drop=True)
doe_matrix_low_confidence = doe_matrix_flagged[doe_matrix_flagged["low_cycle_flag"] == True].reset_index(drop=True)
doe_matrix_low_confidence.to_csv("low_confidence_cells_log.csv", index=False)
print(f"{len(doe_matrix_low_confidence)} cell(s) saved to low_confidence_cells_log.csv for later review")

## cheap check: does ICE have a systematic relationship with the total number of cycles the cell ran? If so, the low-cycle-count filter could be removing specific formulations (selection bias) rather than just random assay noise.
cycle_ice_correlation, cycle_ice_p_value = spearmanr(doe_matrix_flagged["n_cycles"], doe_matrix_flagged["ICE"])
print(f"Spearman ICE x n_cycles (DOE, before filtering): rho={cycle_ice_correlation:.3f}, p={cycle_ice_p_value:.4f}")

In [ ]:
## OHE + RBF baseline
result_loo_ohe_rbf = run_loo_cv_no_leakage(doe_matrix_clean, target_column="ICE",
                                            gp_builder=build_rbf_single_task_gp, config_label="OHE+RBF")
r2_ohe_rbf, rmse_ohe_rbf = plot_loo_cv(result_loo_ohe_rbf, target_name="ICE - DOE (OHE + RBF baseline)")

In [ ]:
## OHE + Matern (isolates the kernel-family effect from the encoding effect)
result_loo_ohe_matern = run_loo_cv_no_leakage(doe_matrix_clean, target_column="ICE",
                                               gp_builder=build_matern_single_task_gp, config_label="OHE+Matern")
r2_ohe_matern, rmse_ohe_matern = plot_loo_cv(result_loo_ohe_matern, target_name="ICE - DOE (OHE + Matern)")

In [ ]:
## Hamming + Matern (mixed kernel)
result_loo_mixed = run_loo_cv_mixed_no_leakage(doe_matrix_clean, target_column="ICE")
r2_mixed, rmse_mixed = plot_loo_cv(result_loo_mixed, target_name="ICE - DOE (Hamming + Matern, mixed kernel)")

In [ ]:
print("Kernel ablation -- DOE (n={}):".format(len(doe_matrix_clean)))
print(f"  OHE + RBF (baseline):      R2={r2_ohe_rbf:.3f}, RMSE={rmse_ohe_rbf:.3f}")
print(f"  OHE + Matern:              R2={r2_ohe_matern:.3f}, RMSE={rmse_ohe_matern:.3f}")
print(f"  Hamming + Matern (mixed):  R2={r2_mixed:.3f}, RMSE={rmse_mixed:.3f}")

print()
print(f"doe: {len(doe_matrix_clean)} cells (after the low-cycle-count filter)")
print(f"if: {len(extra1_matrix)} cells")
print(f"EXTRA2: {len(extra2_matrix)} cells")

## 3.7 Retrospective Validation (Hindcasting)

RQ2: because the experimental campaigns had already been completed, a prospective BO
campaign wasn't possible. Instead, BO is evaluated retrospectively -- hindcasting: start
from a small revealed subset of the DOE, let BO pick which of the remaining, already-run
cells to "reveal" next, and see if it beats revealing cells at random.

In [ ]:
## builds the ordinal feature representation for the full (cleaned) DOE dataset -- same representation used by the mixed-kernel GP in Section 3.6
X_doe_ord, y_doe_ord, ordinal_encoder, numeric_bounds_used, target_scalers = build_ordinal_features(doe_matrix_clean)

**The scaler problem.** Every time a new cell is "revealed" the target distribution the
GP was trained on changes slightly, so the `StandardScaler` has to be refit at each
iteration -- but only on the cells revealed *so far*, never on the full dataset, or future
information would leak into earlier iterations. `train_hindcast_gp` does the fit-scaler
+ fit-GP step together, since they always have to happen on exactly the same revealed
subset. Because the scaler changes at every iteration, "best ICE so far" is tracked in
real units (%) throughout the loop below, not in the (constantly-shifting) scaled space -- 
that's the only scale that stays comparable across iterations, seeds, and between BO and
random selection.

In [ ]:
def train_hindcast_gp(y_full_raw, train_indices, X_tensor, cat_dims):
    ## scales the target (using only the points already revealed) and trains the GP -- always done together at each hindcasting iteration, since the scaler must be refit on exactly the same revealed subset the GP is trained on
    y_train_raw = y_full_raw[train_indices]
    y_train_raw_numpy = y_train_raw.numpy().reshape(-1, 1)

    target_scaler = StandardScaler()
    target_scaler.fit(y_train_raw_numpy)

    y_train_scaled_numpy = target_scaler.transform(y_train_raw_numpy)
    y_train_tensor = torch.tensor(y_train_scaled_numpy.flatten()).unsqueeze(-1)

    model = MixedSingleTaskGP(X_tensor, y_train_tensor, cat_dims=cat_dims)
    mll = ExactMarginalLogLikelihood(model.likelihood, model)
    fit_gpytorch_mll(mll, optimizer_kwargs={"options": {"maxiter": 1000}})

    return model, target_scaler, y_train_tensor

**The BO loop itself.** At each iteration, `run_retrospective_validation` retrains the GP
on whatever's been revealed, scores every still-unrevealed cell with **Expected
Improvement** (an acquisition function that favours candidates likely to beat the current
best, weighted by how uncertain the model still is about them), and reveals whichever cell
scores highest. 15 iterations, starting from 10 revealed cells, repeated for 20 random
seeds (only the initial 10 differ between seeds).

In [ ]:
def run_retrospective_validation(X_full, y_full_raw, cat_dims, n_initial=10, n_iterations=15, random_seed=0):
    ## leakage-safe hindcasting: starts from a random initial seed of revealed cells, and the GP chooses (via Expected Improvement) which cell to reveal next, compared against random selection (run_random_baseline below)
    rng = np.random.default_rng(random_seed)
    n_total = X_full.shape[0]
    all_indices = np.arange(n_total)
    rng.shuffle(all_indices)

    train_indices = list(all_indices[:n_initial])
    pool_indices = list(all_indices[n_initial:])

    ## "best so far" is tracked in real ICE (%), not in the scaled space -- the scaler changes at every iteration, so a scaled value from iteration 3 is not comparable to a scaled value from iteration 10. Real ICE is the only scale that stays consistent across iterations, seeds, and between BO and random.
    best_so_far = y_full_raw[train_indices].max().item()
    best_found_trajectory = [best_so_far]

    n_real_iterations = min(n_iterations, len(pool_indices))

    for iteration in range(n_real_iterations):
        X_train_tensor = X_full[train_indices]
        model, target_scaler, y_train_tensor = train_hindcast_gp(y_full_raw, train_indices, X_train_tensor, cat_dims)

        X_pool_tensor = X_full[pool_indices]
        acquisition_function = LogExpectedImprovement(model=model, best_f=y_train_tensor.max())

        model.eval()
        with torch.no_grad():
            acquisition_values = acquisition_function(X_pool_tensor.unsqueeze(1))

        chosen_position = torch.argmax(acquisition_values).item()
        chosen_index = pool_indices[chosen_position]

        train_indices.append(chosen_index)
        pool_indices.remove(chosen_index)

        best_so_far = max(best_so_far, y_full_raw[chosen_index].item())
        best_found_trajectory.append(best_so_far)

        print(f"run_retrospective_validation: iteration {iteration + 1}/{n_real_iterations} done (seed={random_seed})")

    return best_found_trajectory

`run_random_baseline` is the control: same starting point, same number of iterations, but
picks the next cell to reveal uniformly at random instead of by Expected Improvement.

In [ ]:
def run_random_baseline(X_full, y_full_raw, n_initial=10, n_iterations=15, random_seed=0):
    ##baseline without a GP -- no scaler needed, but uses y_full_raw (real ICE) to keep "best so far" on the same unit as BO, for a fair comparison
    rng = np.random.default_rng(random_seed)
    n_total = X_full.shape[0]
    all_indices = np.arange(n_total)
    rng.shuffle(all_indices)

    train_indices = list(all_indices[:n_initial])
    pool_indices = list(all_indices[n_initial:])

    best_so_far = y_full_raw[train_indices].max().item()
    best_found_trajectory = [best_so_far]

    n_real_iterations = min(n_iterations, len(pool_indices))
    for iteration in range(n_real_iterations):
        chosen_position = rng.integers(0, len(pool_indices))
        chosen_index = pool_indices[chosen_position]

        train_indices.append(chosen_index)
        pool_indices.pop(chosen_position)

        best_so_far = max(best_so_far, y_full_raw[chosen_index].item())
        best_found_trajectory.append(best_so_far)

    return best_found_trajectory

In [ ]:
def plot_retrospective_validation(bo_results, random_results, save_path=None):
    ## mean and standard deviation across the 20 seeds -- the mean alone could give an overconfident impression of how much BO beats random
    mean_bo = np.mean(bo_results, axis=0)
    std_bo = np.std(bo_results, axis=0)
    mean_random = np.mean(random_results, axis=0)
    std_random = np.std(random_results, axis=0)
    iterations = list(range(len(mean_bo)))

    fig, ax = plt.subplots(figsize=(8, 6))

    ax.plot(iterations, mean_bo, marker="o", label="BO (Expected Improvement)", color="tab:blue")
    ax.fill_between(iterations, mean_bo - std_bo, mean_bo + std_bo, alpha=0.2, color="tab:blue")

    ax.plot(iterations, mean_random, marker="o", label="Random selection", color="tab:orange")
    ax.fill_between(iterations, mean_random - std_random, mean_random + std_random, alpha=0.2, color="tab:orange")

    ax.set_xlabel("Iteration")
    ax.set_ylabel("Best ICE found so far (%)")
    ax.set_title("Retrospective validation: BO vs random selection (mean +/- 1 SD, 20 seeds)")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=120)
        print(f"Saved plot to {save_path}")
    plt.show()

In [ ]:
## X_full_df drops the "type" column, since within the DOE dataset "type" doesn't vary (all DOE cells are half-cells) -- cat_dims below assumes X_full_df's column order is exactly [binder, solvent, fec, thickness] after that drop, which comes from how build_ordinal_features orders its output. This is checked explicitly rather than assumed silently, since a silent reordering upstream would otherwise fail quietly instead of raising an error.
X_full_df = X_doe_ord.drop(columns=["type"])
expected_column_order = ["binder", "solvent", "fec", "thickness"]
assert X_full_df.columns.tolist() == expected_column_order, (
    f"X_full_df column order changed: expected {expected_column_order}, got {X_full_df.columns.tolist()}. "
    f"cat_dims=[0, 1] below assumes binder and solvent are the first two columns."
)

X_full = torch.tensor(X_full_df.to_numpy())

## ICE in real units (%), aligned to the same index as X_doe_ord
y_full_raw_series = doe_matrix_clean.loc[X_doe_ord.index, "ICE"]
y_full_raw = torch.tensor(y_full_raw_series.to_numpy())

cat_dims = [0, 1]  ## binder, solvent -- see the assertion above

bo_results = []
random_results = []
for seed in range(20):
    bo_trajectory = run_retrospective_validation(X_full, y_full_raw, cat_dims, n_initial=10, n_iterations=15, random_seed=seed)
    random_trajectory = run_random_baseline(X_full, y_full_raw, n_initial=10, n_iterations=15, random_seed=seed)
    bo_results.append(bo_trajectory)
    random_results.append(random_trajectory)
    print(f"run_retrospective_validation: seed {seed + 1}/20 done")

plot_retrospective_validation(bo_results, random_results)

In [ ]:
## mean best-ICE per iteration, BO vs random -- this is the trajectory reported in Section 4.3 (Table/Figure: mean best ICE per iteration, BO vs random baseline)
mean_bo = np.mean(bo_results, axis=0)
mean_random = np.mean(random_results, axis=0)

for iteration in range(len(mean_bo)):
    print(f"iteration {iteration}: BO={mean_bo[iteration]:.4f}, random={mean_random[iteration]:.4f}")

## 3.8 Cross-Group Bayesian Optimisation Benchmark

Phase 3 (BayBE), addressing RQ4: does adding historical data beyond the DOE improve BO
performance, and does that depend on how much is added and how it's spread across the
design space? Five groups are compared: `DOE`, `EXTRA1`, `EXTRA2`, `DOE+EXTRA1`, and
`DOE+EXTRA1+EXTRA2`.

In [ ]:
def build_campaign_from_group(df_group, target_name="ice", minimize=False):
    ## builds the full BayBE campaign for one group: extracts the observed values, builds the parameters, the objective, and the campaign, all in one place. Each group uses ONLY the values observed in its own data (not the union across groups), since e.g. DOE and EXTRA2 don't even overlap in design space. No TaskParameter -- groups differ only in the data available to them, not in a shared parameter structure (no warm-start). CategoricalParameter/NumericalDiscreteParameter in BayBE require at least 2 distinct values -- columns that are constant within the group (e.g. solvent always EC:DEC in EXTRA2) are excluded from the search space, same principle as the fallback already used in build_single_or_mixed_gp on the BoTorch side.
    fec_values_observed = sorted(df_group["fec"].unique().tolist())
    thickness_values_observed = sorted(df_group["thickness"].unique().tolist())
    binder_values_observed = sorted(df_group["binder"].unique().tolist())
    solvent_values_observed = sorted(df_group["solvent"].unique().tolist())

    parameter_list = []
    columns_for_searchspace = []

    if len(binder_values_observed) >= 2:
        parameter_list.append(CategoricalParameter(name="binder", values=binder_values_observed, encoding="OHE"))
        columns_for_searchspace.append("binder")
    else:
        print(f"build_campaign_from_group: binder constant ({binder_values_observed}), excluded from search space")

    if len(solvent_values_observed) >= 2:
        parameter_list.append(CategoricalParameter(name="solvent", values=solvent_values_observed, encoding="OHE"))
        columns_for_searchspace.append("solvent")
    else:
        print(f"build_campaign_from_group: solvent constant ({solvent_values_observed}), excluded from search space")

    if len(fec_values_observed) >= 2:
        parameter_list.append(NumericalDiscreteParameter(name="fec", values=fec_values_observed, tolerance=0.0))
        columns_for_searchspace.append("fec")
    else:
        print(f"build_campaign_from_group: fec constant ({fec_values_observed}), excluded from search space")

    if len(thickness_values_observed) >= 2:
        parameter_list.append(NumericalDiscreteParameter(name="thickness", values=thickness_values_observed, tolerance=0.0))
        columns_for_searchspace.append("thickness")
    else:
        print(f"build_campaign_from_group: thickness constant ({thickness_values_observed}), excluded from search space")

    if len(parameter_list) == 0:
        raise ValueError("build_campaign_from_group: no parameter with enough variation, group cannot be optimised")

    subspace = SubspaceDiscrete.from_dataframe(df_group[columns_for_searchspace], parameters=parameter_list)
    searchspace = SearchSpace(discrete=subspace)

    target = NumericalTarget(name=target_name, minimize=minimize)
    objective = SingleTargetObjective(target=target)

    return Campaign(searchspace=searchspace, objective=objective)

Two choices worth flagging here. First, each group's candidate space only uses the
parameter values *actually observed in that group's own data* -- no shared `TaskParameter`,
no warm-start between groups, so groups differ only in which data they have, not in a
shared parameter structure. Second, this uses BayBE's **default** GP surrogate, without a
custom kernel: the isolated ablation in Section 3.6 showed that swapping kernel family
alone (without also changing the categorical encoding) gives no real improvement over
OHE+RBF, and BayBE has no built-in Hamming-kernel treatment to plug in instead -- so the
gain from Section 3.6's mixed kernel wasn't expected to carry over here, and the default
was kept rather than forcing a mismatched kernel choice.

Next, `prepare_measurements` turns a group's `master_log` rows into the plain
binder/solvent/fec/thickness/ice table BayBE expects.

In [ ]:
def prepare_measurements(master_log_df, binder_column="binder", solvent_column="solvent",
                          fec_column="fec", thickness_column="thickness", ice_column="ICE",
                          fec_round_decimals=3, thickness_round_decimals=0,
                          valid_ice_range=(0.0, 100.0)):
    ## renames and rounds fec/thickness BEFORE any exact-equality comparison in BayBE -- avoids "==" match failures from floating-point precision differences between sources. Also filters out ICE outside the physically plausible range (0-100%), same criterion already used in filter_plausible_ice/FeatureTransformer, so a known measurement error (e.g. IF2007E5, ICE~2496%) can't contaminate the known optimum or BayBE's regret calculation.
    measurements_df = master_log_df.copy()

    rename_map = {
        binder_column: "binder", solvent_column: "solvent",
        fec_column: "fec", thickness_column: "thickness", ice_column: "ice",
    }
    measurements_df = measurements_df.rename(columns=rename_map)

    measurements_df["fec"] = measurements_df["fec"].round(fec_round_decimals)
    measurements_df["thickness"] = measurements_df["thickness"].round(thickness_round_decimals)

    measurements_df = measurements_df[["binder", "solvent", "fec", "thickness", "ice"]]
    measurements_df = measurements_df.dropna(subset=["ice"])

    is_in_range = (measurements_df["ice"] >= valid_ice_range[0]) & (measurements_df["ice"] <= valid_ice_range[1])
    n_out_of_range = (~is_in_range).sum()
    if n_out_of_range > 0:
        print(f"prepare_measurements: {n_out_of_range} row(s) with ice outside {valid_ice_range}, excluded")
    measurements_df = measurements_df[is_in_range].reset_index(drop=True)

    return measurements_df

In [ ]:
def build_measurement_groups(group_sources, groupby_columns=["binder", "solvent", "fec", "thickness"],
                              target_column="ice", agg="mean"):
    ## builds the BayBE group dictionary from a {group_name: [list of measurement dataframes to combine]} dictionary -- concatenates each group's sources and aggregates replicates at the same design point (otherwise BayBE would find multiple rows for the same combination). Replaces manually repeating concat+groupby for each of the 5 groups.
    baybe_groups = {}
    for group_name, measurement_list in group_sources.items():
        combined = pd.concat(measurement_list, axis=0, ignore_index=True)
        deduplicated = combined.groupby(groupby_columns, as_index=False)[target_column].agg(agg)
        baybe_groups[group_name] = deduplicated
        print(f"build_measurement_groups: group {group_name} has {len(deduplicated)} unique design point(s)")

    return baybe_groups

In [ ]:
def run_group_backtest(group_name, df_group, batch_size=2, n_doe_iterations=10, n_mc_iterations=10, random_seed=0):
    campaign = build_campaign_from_group(df_group)
    scenario_dict = {group_name: campaign}

    result_df = simulate_scenarios(
        scenario_dict, df_group, batch_size=batch_size,
        n_doe_iterations=n_doe_iterations, n_mc_iterations=n_mc_iterations, random_seed=random_seed,
    )
    print(f"run_group_backtest: group {group_name} finished")
    return result_df


In [ ]:
def run_all_group_backtests(baybe_groups, batch_size=2, n_doe_iterations=10, n_mc_iterations=10, random_seed=0):
    result_list = []
    for group_name, df_group in baybe_groups.items():
        group_result_df = run_group_backtest(
            group_name=group_name, df_group=df_group, batch_size=batch_size,
            n_doe_iterations=n_doe_iterations, n_mc_iterations=n_mc_iterations, random_seed=random_seed,
        )
        result_list.append(group_result_df)

    return pd.concat(result_list, axis=0, ignore_index=True)

In [ ]:
def suggest_backtest_sizing(group_size, seed_fraction=0.2, batch_size=1):
    ## converts a group size into a seed size + number of BO iterations, given a fixed seed fraction and batch size -- used to scale the experimental budget relative to whichever group is used as the sizing reference (see the three scenarios below)
    seed_size = int(round(group_size * seed_fraction))
    remaining_size = group_size - seed_size

    if remaining_size <= 0:
        raise ValueError("group_size too small for this seed_fraction -- reduce seed_fraction")

    n_doe_iterations = remaining_size // batch_size

    return {
        "group_size": group_size, "seed_size": seed_size, "remaining_size": remaining_size,
        "batch_size": batch_size, "n_doe_iterations": n_doe_iterations,
    }

In [ ]:
def plot_regret_by_group(all_results, regret_column="regret", iteration_column="Iteration",
                          scenario_column="Scenario", save_path=None):
    ## compares the mean regret trajectory across the groups, on the same iteration scale (since the budget was normalised relative to the smallest group)
    fig, ax = plt.subplots(figsize=(8, 6))

    for group in sorted(all_results[scenario_column].unique().tolist()):
        df_group = all_results[all_results[scenario_column] == group]
        summary = df_group.groupby(iteration_column, as_index=False)[regret_column].mean()
        ax.plot(summary[iteration_column], summary[regret_column], marker="o", label=group)

    ax.set_xlabel("Iteration")
    ax.set_ylabel("Regret (global optimum - best found)")
    ax.set_title("Regret comparison across BO groups (normalised budget)")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=120)
        print(f"Saved plot to {save_path}")
    plt.show()

In [ ]:
def summarize_group_convergence(group_name, df_group, all_results, threshold_pct=90.0,
                                 cumbest_column="ice_CumBest", target_column="ice",
                                 scenario_column="Scenario", iteration_column="Iteration"):
    ## averages the cumulative best across the 20 MC repetitions first, per iteration, and only then checks the threshold -- same principle already used in plot_regret_by_group. An earlier version checked the threshold row by row across all repetitions mixed together and took the min, so it picked up the first time ANY single repetition happened to cross the threshold. With a small seed budget (scaled from the smallest group), one lucky draw out of 20 was enough to give iteration 0 to almost every group, so that version wasn't usable.
    own_optimum = df_group[target_column].max()

    group_results = all_results[all_results[scenario_column] == group_name].copy()

    ## averages the MC repetitions per iteration -- one row per iteration, with the mean cumulative-best ICE at that iteration
    mean_cumbest_per_iteration = group_results.groupby(iteration_column, as_index=False)[cumbest_column].mean()
    mean_cumbest_per_iteration["pct_of_own_best"] = (
        mean_cumbest_per_iteration[cumbest_column] / own_optimum * 100.0
    )

    above_threshold = mean_cumbest_per_iteration[mean_cumbest_per_iteration["pct_of_own_best"] >= threshold_pct]
    iterations_to_threshold = above_threshold[iteration_column].min() if len(above_threshold) > 0 else None

    return {
        "group": group_name, "n_design_points": len(df_group),
        "own_optimum_ice": own_optimum, "iterations_to_90pct": iterations_to_threshold,
    }


**Why this metric was tried first, and why it was dropped.** "How many iterations to
reach 90% of the group's own optimum" is intuitive, but even after fixing the bug above it
turned out uninformative for `EXTRA2` specifically: its own optimum sits low enough,
relative to where the search starts, that the 90% threshold gets crossed almost
immediately regardless of how well BO is actually searching -- so the metric can't
distinguish an efficient search from a lucky one for that group. Since the threshold-based
comparison stopped being meaningful for the exact groups we most wanted to compare, we
switched to `summarize_final_regret` below: the regret at the last iteration, with no
threshold to cross, so it can't have this problem for any group regardless of its own
optimum. `summarize_final_regret` is what Table 4/Section 4.5 actually reports; the check
below is kept as a record of that decision, not as a result.

In [ ]:
def summarize_final_regret_dual(all_results, baybe_groups, known_optimum_global,
                                 ice_column="ice_CumBest", iteration_column="Iteration",
                                 scenario_column="Scenario"):
    ## expanded version of summarize_final_regret: regret against the global optimum (as before) AND against each group's own optimum, side by side -- makes explicit how much of a group's high regret comes from its own physical ceiling (e.g. full-cell) versus from BO's efficiency within that group Also reports the standard deviation across the 20 Monte Carlo repetitions at the final iteration. Since known_optimum_global and each group's own_optimum are fixed constants (not random), the standard deviation of the regret is the same as the standard deviation of mean_final_cumbest itself, whether measured against the global or the own optimum -- so one std column covers both regret columns.
    own_optimums = compute_group_own_optimum(baybe_groups)

    summary_rows = []
    for group in sorted(all_results[scenario_column].unique().tolist()):
        group_results = all_results[all_results[scenario_column] == group]
        last_iteration = group_results[iteration_column].max()
        last_iteration_results = group_results[group_results[iteration_column] == last_iteration]

        mean_final_cumbest = last_iteration_results[ice_column].mean()
        std_final_cumbest = last_iteration_results[ice_column].std()
        regret_vs_global = known_optimum_global - mean_final_cumbest
        regret_vs_own = own_optimums[group] - mean_final_cumbest

        summary_rows.append({
            "group": group, "own_optimum": own_optimums[group], "mean_final_cumbest": mean_final_cumbest,
            "regret_vs_global": regret_vs_global, "regret_vs_own": regret_vs_own,
            "std_regret": std_final_cumbest,
        })

    df_summary = pd.DataFrame(summary_rows)
    return df_summary.sort_values("regret_vs_own").reset_index(drop=True)

### Preparing the five groups

`DOE`, `EXTRA1`, and `EXTRA2` measurements are prepared individually, then combined into the
five groups compared in the benchmark: the three primary groups plus `doe_if` and
`DOE+EXTRA1+EXTRA2`.

In [ ]:
doe_measurements = prepare_measurements(doe_matrix_raw, ice_column="ICE")
extra1_measurements = prepare_measurements(extra1_matrix, ice_column="ICE")
extra2_measurements = prepare_measurements(extra2_matrix, ice_column="ICE")

group_sources = {
    "DOE": [doe_measurements],
    "EXTRA1": [extra1_measurements],
    "EXTRA2": [extra2_measurements],
    "DOE+EXTRA1": [doe_measurements, extra1_measurements],
    "DOE+EXTRA1+EXTRA2": [doe_measurements, extra1_measurements, extra2_measurements],
}
baybe_groups = build_measurement_groups(group_sources)

In [ ]:
def compute_group_own_optimum(baybe_groups, target_column="ice"):
    ## best ICE within each group, separate from the global known optimum (best across all groups) -- used to compare regret against a ceiling each group can actually reach, rather than a ceiling some groups (e.g. EXTRA2, full-cell) structurally cannot reach, given the lower full-cell ICE documented in Section 4.1
    return {group_name: df_group[target_column].max() for group_name, df_group in baybe_groups.items()}


### Scenario 1 -- original sizing (budget set relative to `EXTRA2`, the smallest group)

All five groups are compared, with the experimental budget for every backtest scaled
relative to `EXTRA2` (5 unique design points) -- the smallest group overall.

In [ ]:
group_sizes = {group_name: len(df_group) for group_name, df_group in baybe_groups.items()}
smallest_group_name = min(group_sizes, key=group_sizes.get)
sizing_original = suggest_backtest_sizing(group_size=group_sizes[smallest_group_name], seed_fraction=0.2, batch_size=2)
print(f"Budget normalised by the smallest group ({smallest_group_name}): {sizing_original}")

all_results_original = run_all_group_backtests(
    baybe_groups, batch_size=sizing_original["batch_size"],
    n_doe_iterations=sizing_original["n_doe_iterations"], n_mc_iterations=20,
)

known_optimum_original = pd.concat([doe_measurements, extra1_measurements, extra2_measurements], ignore_index=True)["ice"].max()
all_results_original["regret"] = known_optimum_original - all_results_original["ice_CumBest"]

plot_regret_by_group(all_results_original)
final_regret_original = summarize_final_regret_dual(all_results_original, baybe_groups, known_optimum_original)
print("Scenario 1 (original, sizing by EXTRA2) -- final regret by group:")
print(final_regret_original)

As a record of the reasoning above: here's what `summarize_group_convergence` actually
gives for this scenario, run on `EXTRA2` specifically. The mean cumulative-best already
starts close to (or above) the 90% threshold at iteration 0, before BO has done anything -- 
confirming that the 90% threshold is too easy to clear for this group's ICE range, and
that `iterations_to_90pct` can't tell an efficient search apart from a lucky start here.

In [ ]:
convergence_summary = []
for group_name, df_group in baybe_groups.items():
    convergence_summary.append(summarize_group_convergence(group_name, df_group, all_results_original, threshold_pct=90.0))

convergence_summary_df = pd.DataFrame(convergence_summary)
print(convergence_summary_df)

## diagnostic: EXTRA2's own mean cumulative-best (%) at each iteration -- if it's already close to 100% at iteration 0, that's direct confirmation of the reasoning above
extra2_group = baybe_groups["EXTRA2"]
extra2_own_optimum = extra2_group["ice"].max()

extra2_results = all_results_original[all_results_original["Scenario"] == "EXTRA2"].copy()
extra2_mean_cumbest = extra2_results.groupby("Iteration", as_index=False)["ice_CumBest"].mean()
extra2_mean_cumbest["pct_of_own_best"] = extra2_mean_cumbest["ice_CumBest"] / extra2_own_optimum * 100.0

print(f"\nEXTRA2's own optimum: {extra2_own_optimum}")
print(extra2_mean_cumbest)

### Scenario 2 -- sizing excludes `EXTRA2` (all five groups still compared)

The budget is now set relative to the smallest of the *other four* groups (`EXTRA1`), so `EXTRA2`
still participates in the comparison but no longer constrains everyone else's budget.

In [ ]:
group_sizes_excl_extra2 = {name: size for name, size in group_sizes.items() if name != "EXTRA2"}
smallest_excl_extra2 = min(group_sizes_excl_extra2, key=group_sizes_excl_extra2.get)
sizing_excl_extra2 = suggest_backtest_sizing(group_size=group_sizes_excl_extra2[smallest_excl_extra2], seed_fraction=0.2, batch_size=2)
print(f"Scenario 2 -- budget normalised excluding EXTRA2 from sizing, by the smallest remaining group "
      f"({smallest_excl_extra2}): {sizing_excl_extra2}")

all_results_excl_extra2_sizing = run_all_group_backtests(
    baybe_groups, batch_size=sizing_excl_extra2["batch_size"],
    n_doe_iterations=sizing_excl_extra2["n_doe_iterations"], n_mc_iterations=20,
)

known_optimum_excl_extra2_sizing = known_optimum_original  ## same global optimum, all 5 groups still compared
all_results_excl_extra2_sizing["regret"] = known_optimum_excl_extra2_sizing - all_results_excl_extra2_sizing["ice_CumBest"]

plot_regret_by_group(all_results_excl_extra2_sizing)

final_regret_excl_extra2_sizing = summarize_final_regret_dual(all_results_excl_extra2_sizing, baybe_groups, known_optimum_original)
print("Scenario 2 (sizing excludes EXTRA2, 5 groups compared) -- final regret by group:")
print(final_regret_excl_extra2_sizing)

### Scenario 3 -- `EXTRA2` removed entirely (3 groups: `DOE`, `EXTRA1`, `DOE+EXTRA1`)

`DOE+EXTRA1+EXTRA2` is dropped along with `EXTRA2`, since it no longer makes sense without `EXTRA2`.
This checks whether Scenario 2's change came from the larger budget itself, or from
`EXTRA2`'s continued presence in the comparison.

In [ ]:
group_sources_no_extra2 = {
    "DOE": [doe_measurements],
    "EXTRA1": [extra1_measurements],
    "DOE+EXTRA1": [doe_measurements, extra1_measurements],
}
baybe_groups_no_extra2 = build_measurement_groups(group_sources_no_extra2)

group_sizes_no_extra2 = {name: len(df_group) for name, df_group in baybe_groups_no_extra2.items()}
smallest_no_extra2 = min(group_sizes_no_extra2, key=group_sizes_no_extra2.get)
sizing_no_extra2 = suggest_backtest_sizing(group_size=group_sizes_no_extra2[smallest_no_extra2], seed_fraction=0.2, batch_size=2)
print(f"Scenario 3 -- budget normalised (EXTRA2 out of the comparison), by the smallest group "
      f"({smallest_no_extra2}): {sizing_no_extra2}")

all_results_no_extra2 = run_all_group_backtests(
    baybe_groups_no_extra2, batch_size=sizing_no_extra2["batch_size"],
    n_doe_iterations=sizing_no_extra2["n_doe_iterations"], n_mc_iterations=20,
)

known_optimum_no_extra2 = pd.concat([doe_measurements, extra1_measurements], ignore_index=True)["ice"].max()
all_results_no_extra2["regret"] = known_optimum_no_extra2 - all_results_no_extra2["ice_CumBest"]

plot_regret_by_group(all_results_no_extra2)

final_regret_no_extra2 = summarize_final_regret_dual(all_results_no_extra2, baybe_groups_no_extra2, known_optimum_original)
print("Scenario 3 (EXTRA2 removed, 3 groups) -- final regret by group:")
print(final_regret_no_extra2)

### Comparing the three scenarios (Table 4)

In [ ]:
def build_scenario_comparison_table(summary_original, summary_scenario2, summary_scenario3):
    ## combines the three final-regret summaries (one per sizing scenario) into a single long table, with a "scenario" column identifying where each row came from
    summary_original = summary_original.copy()
    summary_original["scenario"] = "1: original (sizing by EXTRA2)"

    summary_scenario2 = summary_scenario2.copy()
    summary_scenario2["scenario"] = "2: sizing excludes EXTRA2, 5 groups"

    summary_scenario3 = summary_scenario3.copy()
    summary_scenario3["scenario"] = "3: EXTRA2 removed, 3 groups"

    combined = pd.concat([summary_original, summary_scenario2, summary_scenario3], ignore_index=True)
    combined = combined[["scenario", "group", "own_optimum", "mean_final_cumbest",
                          "regret_vs_global", "regret_vs_own", "std_regret"]]
    return combined.sort_values(["group", "scenario"]).reset_index(drop=True)


In [ ]:
def build_sizing_comparison_table(sizing_original, sizing_shared):
    ## small helper table making explicit which budget (n_doe_iterations, seed_size) was used in each scenario -- explains why the regret numbers change between them
    rows = [
        {"scenario": "1: original (sizing by EXTRA2)", "reference_group": "EXTRA2",
         "seed_size": sizing_original["seed_size"], "n_doe_iterations": sizing_original["n_doe_iterations"],
         "batch_size": sizing_original["batch_size"]},
        {"scenario": "2/3: sizing excludes EXTRA2", "reference_group": "EXTRA1",
         "seed_size": sizing_shared["seed_size"], "n_doe_iterations": sizing_shared["n_doe_iterations"],
         "batch_size": sizing_shared["batch_size"]},
    ]
    return pd.DataFrame(rows)

In [ ]:
scenario_comparison_table = build_scenario_comparison_table(
    final_regret_original, final_regret_excl_extra2_sizing, final_regret_no_extra2,
)
print("Final-regret comparison across sizing scenarios:")
print(scenario_comparison_table)

print()
sizing_comparison_table = build_sizing_comparison_table(sizing_original, sizing_excl_extra2)
print("Budget used in each scenario:")
print(sizing_comparison_table)

### Regret against each group's own optimum (Table 4, right-hand column)

The regret figures above are all against a single global optimum, shared by every group.
This is not the fairest comparison for `EXTRA2` (full-cell), which cannot realistically
approach a half-cell optimum for the reasons discussed in Section 4.1. The dual comparison
below reports regret against each group's own best observed ICE as well.

In [ ]:
dual_regret_original = summarize_final_regret_dual(all_results_original, baybe_groups, known_optimum_original)
print("Scenario 1 (original, sizing by EXTRA2):")
print(dual_regret_original)

dual_regret_excl_extra2_sizing = summarize_final_regret_dual(all_results_excl_extra2_sizing, baybe_groups, known_optimum_excl_extra2_sizing)
print("Scenario 2 (sizing excludes EXTRA2, 5 groups compared):")
print(dual_regret_excl_extra2_sizing)

dual_regret_no_extra2 = summarize_final_regret_dual(all_results_no_extra2, baybe_groups_no_extra2, known_optimum_no_extra2)
print("Scenario 3 (EXTRA2 removed, 3 groups):")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print(dual_regret_no_extra2)

## Supplementary figure: design-space coverage across groups (SI-2)

Shows where each dataset group (`DOE` / `EXTRA1` / `EXTRA2`) falls in the FEC x binder plane,
using marker size to indicate coating thickness. Not part of the main analysis pipeline,
kept separately since it's only used for a Supplementary Information figure.

In [ ]:
def plot_design_space_coverage(df_matrix, fec_column="fec", binder_column="binder",
                                thickness_column="thickness", group_column="dataset_group",
                                jitter_strength=0.08, save_path=None):
    ## maps binder to a numeric y-position, with jitter so points with the same binder and fec don't overlap
    binder_positions = {"CMC": 0, "PVDF": 1}
    group_colors = {"DOE": "tab:green", "EXTRA1": "tab:blue", "EXTRA2": "tab:orange"}

    thickness_min = df_matrix[thickness_column].min()
    thickness_max = df_matrix[thickness_column].max()

    fig, ax = plt.subplots(figsize=(9, 5))

    for group in sorted(df_matrix[group_column].dropna().unique().tolist()):
        subset = df_matrix[df_matrix[group_column] == group]

        y_base = subset[binder_column].map(binder_positions)
        noise = np.random.uniform(-jitter_strength, jitter_strength, size=len(subset))
        y_jitter = y_base + noise

        ## normalizes thickness to a readable marker-size range (30 to 220)
        if thickness_max > thickness_min:
            marker_sizes = 30 + (subset[thickness_column] - thickness_min) / (thickness_max - thickness_min) * 190
        else:
            marker_sizes = 100

        ax.scatter(subset[fec_column], y_jitter, s=marker_sizes, color=group_colors.get(group, "gray"),
                   alpha=0.6, edgecolors="black", linewidths=0.5, label=group)

    ax.set_yticks([0, 1])
    ax.set_yticklabels(["CMC", "PVDF"])
    ax.set_xlabel("FEC (wt%)")
    ax.set_ylabel("Binder")
    ax.set_title("Design-space coverage across groups (FEC x binder, marker size = thickness)")
    ax.legend(title="Group")
    ax.grid(True, alpha=0.3)

    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

    return fig, ax

In [ ]:
plot_design_space_coverage(global_matrix)

In [ ]:
plot_pca_scores(scores_global, metadata_global, "dataset_group", save_path="fig_s10_pca_by_group.png")
plot_regret_by_group(all_results_excl_extra2_sizing, save_path="fig_s11_regret_scenario_a.png")